<a href="https://colab.research.google.com/github/fezao12/masters/blob/main/Critical_materials_cientometrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated')
from google.colab import drive
drive.mount('/content/drive/')

Authenticated
Mounted at /content/drive/


#Setting folder

In [ ]:
#Fer's path
folder = '/content/drive/MyDrive/cientometria_resultados/Resultados preliminares/'

#GRUPO: Conceitos + Contexto

##Coleta de dados da OpenAlex

###Extração de IDs



In [ ]:
from google.cloud import bigquery
from google.colab import auth
auth.authenticate_user()

client = bigquery.Client(project="insyspo")

In [ ]:
context = ["energy transition", "renewable energy","energy expansion", "energy system expansion", "electricity expansion", "energy planning",
           "energy demand", "energy supply", "energy infrastructure", "energy planning"]
problems = []
concepts = [
    "critical mineral",
    "critical metal",
    "critical material",
    "critical raw material",
    "strategic mineral",
    "strategic material",
    "key raw material",
    "rare metal",
    "green transition mineral",
    "green transition metal",
    "green transition material",
    "green transition raw material",
    #"strategic metal",
    "rare earth element",
    "battery metal",
    #"transition metal",
    #"mineral resource",
    "raw material dependenc",
    # "high-demand mineral",
    # "high-demand metal",
    #"material scarci",
    #"resource scarci",
    # "metal sustainab",
    #"mineral sustainab",
    #"resource critical",
    #"material critical",
    "aluminium", "antimony", "arsenic", "barite", "beryllium",
    "bismuth", "cerium", "cesium", "chromium", "cobalt", "dysprosium", "erbium", "europium", "fluorspar",
    "gadolinium", "gallium", "germanium", "graphite", "hafnium", "holmium", "indium", "iridium",
    "lanthanum", "lithium", "lutetium",
    #"magnesium",
    "manganese", "neodymium", "nickel", "niobium",
    "palladium", "platinum", "praseodymium", "rhodium", "rubidium", "ruthenium", "samarium",
    "scandium", "tantalum", "tellurium", "terbium", "thulium", "titanium", "tungsten",
    "uranium", "vanadium", "ytterbium", "yttrium", "zinc", "zirconium", "copper",
    #"clean tech",
    #"critical supply chain",
    #"supply chain vulnerability",
    "rare earth",
    #"battery supply chain",
    "energy storage material"
    #"strategic resource",
    #"low-carbon technology",
    # "raw material governance",
    #"sustainable mining",
    #"circular mining",
    #"secondary sourcing",
    #"geopolitical risk",
    # "critical material recycling",
    #"resource diplomacy",
    #"local content policy",
    #"net-zero strategy",
    #"carbon-neutral technology",
    #"renewable infrastructure"
]

def build_like_clauses(terms, field):
    if not terms:
        return ""
    return "\n                OR ".join([f"LOWER({field}) LIKE '%{term.lower()}%'" for term in terms])

# Build clauses
problem_display_clause = build_like_clauses(problems, "display_name")
concept_display_clause = build_like_clauses(concepts, "display_name")
context_display_clause = build_like_clauses(context, "display_name")

problem_abstract_clause = build_like_clauses(problems, "abstract")
concept_abstract_clause = build_like_clauses(concepts, "abstract")
context_abstract_clause = build_like_clauses(context, "abstract")

def make_where_clause(problem_clause, concept_clause, context_clause):
    clauses = []
    if problem_clause:
        clauses.append(f"({problem_clause})")
    if concept_clause:
        clauses.append(f"({concept_clause})")
    if context_clause:
        clauses.append(f"({context_clause})")
    return " AND ".join(clauses) if clauses else "TRUE"

works_where = make_where_clause(problem_display_clause, concept_display_clause, context_display_clause)
abstract_where = make_where_clause(problem_abstract_clause, concept_abstract_clause, context_abstract_clause)

query = f"""
DROP TABLE IF EXISTS `multiobs.userdb_fernando_palomari_spolidorio.cm_work_ids_3`;

CREATE TABLE `insyspo.userdb_fernando_spolidorio.cm_work_ids_3` AS (
    SELECT DISTINCT id
    FROM (
        SELECT id
        FROM `insyspo.publicdb_openalex_2024_10_rm.works`
        WHERE {works_where}
        UNION ALL
        SELECT id
        FROM `insyspo.projectdb_cwts_summer_school_2023.works_abstracts`
        WHERE {abstract_where}
    )
);
"""


In [ ]:
# Execute the query
job = client.query(query)
job.result()  # Wait for completion
print("Query completed and table created.")

Query completed and table created.


In [ ]:
%%bigquery work_ids_3 --project multiobs

select * from multiobs.userdb_fernando_palomari_spolidorio.cm_work_ids_3

AttributeError: partially initialized module 'bigframes' has no attribute 'dtypes' (most likely due to a circular import)

In [ ]:
len(work_ids_3.drop_duplicates(subset=['id']))

15082

###Filtros

####Artigos e anos

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS (

SELECT a.id, b.doi, b.title, b.type, b.publication_year
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works AS b ON a.id = b.id
WHERE b.type = 'article' --and b.publication_year > 2000 and b.publication_year < 2025
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Volume de trabalhos que são artigos.
len(work_ids_a_3.drop_duplicates(subset=['id']))

10887

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS (

SELECT a.id, b.doi, b.title, b.type, b.publication_year
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works AS b ON a.id = b.id
WHERE b.type = 'article' and b.publication_year >= 2000 and b.publication_year < 2024
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Volume de trabalhos que foram publicados entre 2000 e 2025
work_ids_a_3 = work_ids_a_3[
    (work_ids_a_3['publication_year'] > 2000) &
    (work_ids_a_3['publication_year'] < 2025)
]

len(work_ids_a_3)

8683

####Maxscores

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS (

WITH MaxScores AS (
    SELECT a.*, MAX(b.score) AS max_score
    FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS a
    JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
    GROUP BY all
)

select * from maxscores

)


Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_max_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Artigos pós-dedupe
len(work_ids_a_max_3.drop_duplicates(subset=['id']))

8668

###Fields & subfields

In [ ]:
%%bigquery work_ids_a_max_fs_3 --project insyspo

SELECT a.*, f.abstract as abstract, c.display_name as topic, e.display_name AS field, d.display_name as subfield, i.country, h.work_id,
institution_id, h.author_id, j.display_name as author, k.funder_id, l.display_name as funder, i.display_name as institution
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS a
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.topics AS c ON b.topic_id = c.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.subfields AS d ON c.subfield = d.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.fields AS e ON c.field = e.id
LEFT JOIN insyspo.projectdb_cwts_summer_school_2023.works_abstracts f ON a.id = f.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works g ON a.id = g.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_authorships AS h ON a.id = h.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.institutions AS i ON h.institution_id = i.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.authors AS j ON h.author_id = j.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_grants AS k ON a.id = k.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.funders AS l ON k.funder_id = l.id



Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Verificando se o volume total de artigos está correto (precisa ser exatamente igual ao volume antes da adição de fields e subfields).
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

8668

In [ ]:
work_ids_a_max_fs_3[['id','doi','title','topic','field','subfield','country']].head()

,id,doi,title,topic,field,subfield,country
0,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States
1,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,Electrochemical Detection of Heavy Metal Ions,Chemistry,Electrochemistry,United States
2,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",Australia
3,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,Electrochemical Detection of Heavy Metal Ions,Chemistry,Electrochemistry,Australia
4,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States


## Análise de palavras-chaves identificadas

###Criação de colunas

####Criação das colunas problems_kw e concept_kw

In [ ]:
import pandas as pd
import re

# --- 1. Exceções para match exato ---
exact_match_words = {"tin", "lead"}

# --- 2. Função de extração robusta (usa set para não duplicar) ---
def extract_keywords(text, keywords, exact_match_words):
    if pd.isna(text):
        return set()
    text = text.lower()
    found = set()
    for kw in keywords:
        kw_lc = kw.lower().strip()
        if kw_lc in exact_match_words:
            if re.search(rf'\b{re.escape(kw_lc)}\b', text):
                found.add(kw)
        else:
            if kw_lc in text:
                found.add(kw)
    return found

# --- 3. Normalizar listas de palavras-chave ---
problems  = [p.lower().strip() for p in problems]
concepts  = [c.lower().strip() for c in concepts]
contexts  = [c.lower().strip() for c in context]

# --- 4. Aplicar no DataFrame, unindo título e abstract SEM duplicar ---
work_ids_a_max_fs_3["problem_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], problems, exact_match_words)
        | extract_keywords(row["abstract"], problems, exact_match_words)
    ),
    axis=1
)

work_ids_a_max_fs_3["concept_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], concepts, exact_match_words)
        | extract_keywords(row["abstract"], concepts, exact_match_words)
    ),
    axis=1
)

work_ids_a_max_fs_3["context_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], contexts, exact_match_words)
        | extract_keywords(row["abstract"], contexts, exact_match_words)
    ),
    axis=1
)

# --- 5. Identificar casos problemáticos ---
no_concept_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["concept_kw"].apply(lambda x: len(x) == 0)
]

no_problem_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["problem_kw"].apply(lambda x: len(x) == 0)
]

no_context_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["context_kw"].apply(lambda x: len(x) == 0)
]

print("⚠️ Registros com ZERO conceitos detectados:", len(no_concept_rows))
print("⚠️ Registros com ZERO problemas detectados:", len(no_problem_rows))
print("⚠️ Registros com ZERO contextos detectados:", len(no_context_rows))


⚠️ Registros com ZERO conceitos detectados: 0
⚠️ Registros com ZERO problemas detectados: 186718
⚠️ Registros com ZERO contextos detectados: 0


####Criação da coluna all_keywords

In [ ]:
work_ids_a_max_fs_3["all_keywords"] = work_ids_a_max_fs_3.apply(
    lambda row: sorted(set(row["problem_kw"] + row["concept_kw"] + row["context_kw"])),
    axis=1)

work_ids_a_max_fs_3[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,2009,0.9666,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[manganese, renewable energy]"
1,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,2009,0.9666,Electrochemical Detection of Heavy Metal Ions,Chemistry,Electrochemistry,United States,"[manganese, renewable energy]"
2,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,2009,0.9666,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",Australia,"[manganese, renewable energy]"
3,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,2009,0.9666,Electrochemical Detection of Heavy Metal Ions,Chemistry,Electrochemistry,Australia,"[manganese, renewable energy]"
4,271715306,None,Inorganic chemistry in service of renewable energy science: solar driven water oxidation by bioinspired manganese catalysts,2009,0.9666,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[manganese, renewable energy]"


In [ ]:
#Salva os resultados no drive, em parquet.
work_ids_a_max_fs_3.to_parquet(folder + 'group_bronze.parquet', index=False)

In [ ]:
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

102

#SHORTCUT GROUP BRONZE

###20 combinações de kw mais comuns por área do conhecimento

In [ ]:
import pandas as pd
folder = '/content/drive/MyDrive/cientometria_resultados/Resultados preliminares/' #Fer's path
work_ids_a_max_fs_3 = pd.read_parquet(folder + 'group_bronze.parquet') #carrega resultados da busca (Conceito+Contexto)


In [ ]:
#Resultados brutos
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

8668

In [ ]:
work_ids_a_max_fs_3[['id','all_keywords']].head()

,id,all_keywords
0,271715306,"[manganese, renewable energy]"
1,271715306,"[manganese, renewable energy]"
2,271715306,"[manganese, renewable energy]"
3,271715306,"[manganese, renewable energy]"
4,271715306,"[manganese, renewable energy]"


In [ ]:
df=work_ids_a_max_fs_3
df["all_keywords"] = df["all_keywords"].apply(lambda x: tuple(x))  # Convert lists to tuples

df_grouped = (
    df.groupby(["field", "all_keywords"])
    .size()
    .reset_index(name="count")
)

# Get top 10 counts per field
df_top10_3 = df_grouped.groupby("field", group_keys=False).apply(lambda x: x.nlargest(20, "count"))

df_top10_3


/tmp/ipykernel_731/3636565974.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_top10_3 = df_grouped.groupby("field", group_keys=False).apply(lambda x: x.nlargest(20, "count"))


,field,all_keywords,count
38,Agricultural and Biological Sciences,"(renewable energy, zinc)",51
22,Agricultural and Biological Sciences,"(energy demand, zinc)",43
30,Agricultural and Biological Sciences,"(energy supply, zinc)",43
28,Agricultural and Biological Sciences,"(energy supply, platinum)",42
18,Agricultural and Biological Sciences,"(copper, renewable energy)",32
...,...,...,...
4030,Social Sciences,"(aluminium, renewable energy)",7
4055,Social Sciences,"(critical mineral, critical raw material, ener...",7
4087,Social Sciences,"(energy transition, indium)",7
4051,Social Sciences,"(copper, renewable energy)",6


###Volume e representatividade de palavras-chaves de Conceitos

In [ ]:
from collections import Counter
import pandas as pd

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)
# Flatten all lists of keywords into one big list
all_keywords_flat = work_ids_a_max_fs_3['concept_kw'].dropna().explode()

# Count keyword frequencies
keyword_counts = Counter(all_keywords_flat)

# Convert to DataFrame
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['keyword', 'count'])

# Compute total count of all keywords
total_keywords = keyword_df['count'].sum()

# Add percentage column
keyword_df['percent'] = (keyword_df['count'] / total_keywords * 100).round(1)

# Sort by most common
keyword_df = keyword_df.sort_values(by='count', ascending=False).reset_index(drop=True)

# Show full result without

keyword_df

,keyword,count,percent
0,lithium,2650,21.2
1,copper,1184,9.5
2,nickel,996,8.0
3,cobalt,761,6.1
4,platinum,728,5.8
...,...,...,...
56,thulium,5,0.0
57,lutetium,2,0.0
58,battery metal,2,0.0
59,holmium,1,0.0


In [ ]:
# --- deps opcionais (1ª vez no Colab) ---
# !pip install dataframe-image python-docx

import math
import pandas as pd
from datetime import date
from pandas.io.formats.style import Styler  # <- correção da tipagem

# ==============================
# CONFIGURAÇÃO
# ==============================
TITLE        = "Tabela X – Palavras-chave mais frequentes"
SOURCE       = f"Fonte: Elaboração própria ({date.today().year})."
MAX_ROWS     = 30   # linhas por tabela/página
OUT_BASENAME = "tabela_keywords"  # base do nome dos arquivos

# ---------------- utilidades ----------------
def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.rename(columns={
        "keyword": "Palavra-chave",
        "count": "Contagem",
        "percent": "Percentual (%)"
    })

def fmt_int_br(x):
    try:
        return f"{int(x):,}".replace(",", ".")
    except Exception:
        return x

def fmt_pct_br(x):
    try:
        return f"{float(x):.1f}%".replace(".", ",")
    except Exception:
        return x

def split_pages(df: pd.DataFrame, max_rows: int = MAX_ROWS):
    n_pages = math.ceil(len(df)/max_rows)
    for i in range(n_pages):
        yield i+1, df.iloc[i*max_rows:(i+1)*max_rows].copy()

# ---------------- estilo ABNT ----------------
def style_abnt(df_page: pd.DataFrame, subtitle: str = "") -> Styler:
    sty = (
        df_page.style
        .hide(axis="index")
        .set_table_styles([
            {'selector': 'table',
             'props': [('border-collapse','collapse'),
                       ('font-family','"Times New Roman", Times, serif'),
                       ('font-size','12pt'),
                       ('color','#000'),
                       ('width','100%'),
                       ('border-top','1pt solid #000'),
                       ('border-bottom','1pt solid #000')]},
            {'selector': 'thead th',
             'props': [('font-weight','bold'),
                       ('text-align','left'),
                       ('border-bottom','1pt solid #000'),
                       ('padding','4pt 6pt')]},
            {'selector': 'tbody td',
             'props': [('padding','3pt 6pt')]},
        ])
        .set_properties(subset=pd.IndexSlice[:, ["Palavra-chave"]], **{'text-align':'left'})
        .set_properties(subset=pd.IndexSlice[:, ["Contagem","Percentual (%)"]], **{'text-align':'right'})
        .format({"Contagem": fmt_int_br, "Percentual (%)": fmt_pct_br})
    )
    caption_html = f"<div style='font-family:Times New Roman;font-size:12pt;margin-bottom:6pt'><b>{TITLE}</b>"
    if subtitle:
        caption_html += f"<br><span style='font-size:10pt'>{subtitle}</span>"
    caption_html += "</div>"
    foot_html = f"<div style='font-family:Times New Roman;font-size:10pt;margin-top:4pt'>{SOURCE}</div>"
    return sty.set_caption(caption_html + foot_html)

# ---------------- exportação principal ----------------
def build_and_save_tables(keyword_df: pd.DataFrame, out_dir: str = "."):
    # ordenação padrão: contagem desc
    df = keyword_df.copy()
    df = df.sort_values("count", ascending=False, kind="stable")
    df = rename_columns(df)

    pages = list(split_pages(df, MAX_ROWS))
    print(f"Gerando {len(pages)} tabela(s).")

    # HTML + PNG
    try:
        import dataframe_image as dfi
        for page_no, df_page in pages:
            sty = style_abnt(df_page, subtitle=f"Página {page_no} de {len(pages)}")
            html_path = f"{out_dir}/{OUT_BASENAME}_p{page_no}.html"
            png_path  = f"{out_dir}/{OUT_BASENAME}_p{page_no}.png"
            sty.to_html(html_path)
            dfi.export(sty, png_path, table_conversion='matplotlib')  # robusto no Colab
            print("Salvo:", html_path)
            print("Salvo:", png_path)
    except Exception as e:
        print("Exporto HTML/PNG pulado. Motivo:", e)

    # DOCX (Word)
    try:
        from docx import Document
        from docx.shared import Pt
        from docx.oxml import OxmlElement
        from docx.oxml.ns import qn
        from docx.enum.text import WD_ALIGN_PARAGRAPH

        def set_cell_border(cell, **kwargs):
            tc = cell._tc
            tcPr = tc.get_or_add_tcPr()
            tcBorders = tcPr.first_child_found_in("w:tcBorders")
            if tcBorders is None:
                tcBorders = OxmlElement('w:tcBorders')
                tcPr.append(tcBorders)
            for edge in ('top', 'left', 'bottom', 'right'):
                if edge in kwargs:
                    edge_data = kwargs.get(edge)
                    tag = 'w:{}'.format(edge)
                    element = tcBorders.find(qn(tag))
                    if element is None:
                        element = OxmlElement(tag)
                        tcBorders.append(element)
                    for key in ["val", "sz", "color", "space"]:
                        if key in edge_data:
                            element.set(qn('w:'+key), str(edge_data[key]))

        doc = Document()
        # título
        p = doc.add_paragraph()
        run = p.add_run(TITLE)
        run.bold = True
        run.font.name = "Times New Roman"
        run.font.size = Pt(12)

        for page_no, df_page in pages:
            if page_no > 1:
                doc.add_paragraph()
            table = doc.add_table(rows=len(df_page)+1, cols=len(df_page.columns))
            table.autofit = True

            # cabeçalho
            for j, col in enumerate(df_page.columns):
                cell = table.cell(0, j)
                cell.text = str(col)
                for r in cell.paragraphs[0].runs:
                    r.font.name = "Times New Roman"
                    r.font.size = Pt(12)
                    r.bold = True

            # linhas
            for i in range(len(df_page)):
                for j, col in enumerate(df_page.columns):
                    cell = table.cell(i+1, j)
                    val = df_page.iloc[i, j]
                    if col == "Contagem":
                        val = fmt_int_br(val)
                    elif col == "Percentual (%)":
                        val = fmt_pct_br(val)
                    cell.text = str(val)
                    for r in cell.paragraphs[0].runs:
                        r.font.name = "Times New Roman"
                        r.font.size = Pt(12)

            # bordas: só superior + inferior
            nrows, ncols = len(df_page)+1, len(df_page.columns)
            for j in range(ncols):
                set_cell_border(table.cell(0, j),
                                top={"val":"single","sz":"8","color":"000000"},
                                bottom={"val":"single","sz":"8","color":"000000"})
                set_cell_border(table.cell(nrows-1, j),
                                bottom={"val":"single","sz":"8","color":"000000"})
            # remover laterais internas
            for i in range(1, nrows-1):
                for j in range(ncols):
                    set_cell_border(table.cell(i, j),
                                    left={"val":"nil"}, right={"val":"nil"})

            # alinhar números à direita
            num_cols = ["Contagem","Percentual (%)"]
            for i in range(1, nrows):
                for j, col in enumerate(df_page.columns):
                    if col in num_cols:
                        table.cell(i, j).paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.RIGHT

        # fonte
        p2 = doc.add_paragraph(SOURCE)
        p2.runs[0].font.name = "Times New Roman"
        p2.runs[0].font.size = Pt(10)

        docx_path = f"{out_dir}/{OUT_BASENAME}.docx"
        doc.save(docx_path)
        print("Salvo:", docx_path)
    except Exception as e:
        print("Exporto DOCX pulado. Motivo:", e)

# ==============================
# EXEMPLOS
# ==============================
# Local:
# build_and_save_tables(keyword_df, out_dir=".")

# Colab + Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# build_and_save_tables(keyword_df, out_dir="/content/drive/MyDrive/cientometria_resultados/tabelas")


###Evolução Temporal das Principais Palavras-chave de Conceito

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ===== Parâmetros =====
keyword_col = 'concept_kw'
TOP_K = 10

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# 1) Explodir keywords e contar por ano + keyword
exploded_df = (
    work_ids_a_max_fs_3[['publication_year', keyword_col]]
    .explode(keyword_col)
    .dropna()
)

keyword_year_counts = (
    exploded_df
    .groupby(['publication_year', keyword_col])
    .size()
    .reset_index(name='count')
)

# 2) Selecionar TOP_K palavras no período
top_keywords = (
    keyword_year_counts
    .groupby(keyword_col)['count']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_K)
    .index.tolist()
)

df_top = keyword_year_counts[keyword_year_counts[keyword_col].isin(top_keywords)].copy()

# 3) Pivot (anos x palavras) e ordenação por total
pivot = (
    df_top
    .pivot_table(index='publication_year', columns=keyword_col, values='count', aggfunc='sum', fill_value=0)
    .sort_index()
)
pivot = pivot[pivot.sum().sort_values(ascending=False).index]

# 4) Marcadores distintos (10 símbolos)
marker_symbols = [
    'circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
    'triangle-left', 'triangle-right', 'cross', 'x', 'star'
]

# 5) Figura (ABNT/DPCT-IG)
fig = go.Figure()

for (col, marker_sym) in zip(pivot.columns, marker_symbols):
    fig.add_trace(
        go.Scatter(
            x=pivot.index.astype(str),             # anos como categorias ordenadas
            y=pivot[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.8, dash='solid'),  # linhas pretas
            marker=dict(
                symbol=marker_sym,
                size=7,
                color='white',                     # preenchimento branco
                line=dict(color='black', width=1)  # contorno preto para contraste PB
            ),
            hovertemplate="Ano: %{x}<br>Frequência: %{y}<extra>"+str(col)+"</extra>"
        )
    )

fig.update_layout(
    title="Evolução temporal das 10 principais palavras-chave de conceito",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=600,
    margin=dict(l=80, r=80, t=80, b=140),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Frequência da palavra-chave",
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True,
        type='linear'
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,    # legenda abaixo
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
pivot

concept_kw,lithium,copper,nickel,cobalt,platinum,zinc,graphite,vanadium,titanium,manganese
publication_year,,,,,,,,,,
2000,0,1,1,1,1,3,0,0,0,2
2001,3,1,1,0,2,1,1,1,0,2
2002,2,3,1,1,0,3,4,0,0,2
2003,3,4,4,0,0,2,2,0,0,1
2004,2,3,2,0,0,3,0,2,1,0
2005,8,5,3,1,3,4,0,0,2,2
2006,5,6,6,3,2,2,3,2,1,1
2007,1,11,5,0,4,3,0,1,5,2
2008,4,12,7,4,3,4,1,3,1,4


###Volume e representatividade de palavras-chaves de Contexto

In [ ]:
from collections import Counter
import pandas as pd

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# Flatten all lists of keywords into one big list
all_keywords_flat = work_ids_a_max_fs_3['context_kw'].dropna().explode()

# Count keyword frequencies
keyword_counts = Counter(all_keywords_flat)

# Convert to DataFrame
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['keyword', 'count'])

# Compute total count of all keywords
total_keywords = keyword_df['count'].sum()

# Add percentage column
keyword_df['percent'] = (keyword_df['count'] / total_keywords * 100).round(1)

# Sort by most common
keyword_df = keyword_df.sort_values(by='count', ascending=False).reset_index(drop=True)

# Show full result without truncation
with pd.option_context('display.max_rows', None):
    print(keyword_df)

keyword_df

                 keyword  count  percent
0       renewable energy   6191     66.6
1          energy demand   1470     15.8
2          energy supply    808      8.7
3      energy transition    717      7.7
4  energy infrastructure     89      1.0
5       energy expansion     13      0.1
6        energy planning     11      0.1
7  electricity expansion      1      0.0


,keyword,count,percent
0,renewable energy,6191,66.6
1,energy demand,1470,15.8
2,energy supply,808,8.7
3,energy transition,717,7.7
4,energy infrastructure,89,1.0
5,energy expansion,13,0.1
6,energy planning,11,0.1
7,electricity expansion,1,0.0


###Evolução Temporal das Principais Palavras-chave de Contexto


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ===== Parâmetros =====
keyword_col = 'context_kw'
TOP_K = 10

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# 1) Explodir keywords e contar por ano + keyword
exploded_df = (
    work_ids_a_max_fs_3[['publication_year', keyword_col]]
    .explode(keyword_col)
    .dropna()
)

keyword_year_counts = (
    exploded_df
    .groupby(['publication_year', keyword_col])
    .size()
    .reset_index(name='count')
)

# 2) Selecionar TOP_K palavras no período
top_keywords = (
    keyword_year_counts
    .groupby(keyword_col)['count']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_K)
    .index.tolist()
)

df_top = keyword_year_counts[keyword_year_counts[keyword_col].isin(top_keywords)].copy()

# 3) Pivot (anos x palavras) e ordenação
pivot = (
    df_top.pivot_table(index='publication_year', columns=keyword_col, values='count', aggfunc='sum', fill_value=0)
         .sort_index()
)
pivot = pivot[pivot.sum().sort_values(ascending=False).index]  # maiores primeiro

# 4) Padrões de linha (apenas traço, sem marcadores)
dash_patterns = [
    'solid', 'dash', 'dot', 'dashdot',
    'longdash', 'longdashdot', 'dash', 'dot', 'dashdot', 'longdash'
]

# 5) Figura Plotly (ABNT/DPCT-IG)
fig = go.Figure()

for col, dash in zip(pivot.columns, dash_patterns):
    fig.add_trace(
        go.Scatter(
            x=pivot.index.astype(str),   # trata anos como categorias ordenadas
            y=pivot[col],
            mode='lines',
            name=str(col),
            line=dict(color='black', width=1.5, dash=dash),
            hovertemplate="Ano: %{x}<br>Frequência: %{y}<extra>"+str(col)+"</extra>"
        )
    )

fig.update_layout(
    title="Evolução temporal das principais palavras-chave de contexto",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=600,
    margin=dict(l=80, r=80, t=80, b=140),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Frequência da palavra-chave",
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,   # legenda abaixo do gráfico
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


###Volume de kw por área

In [ ]:
import pandas as pd

# Step 1: Calculate total count per field
field_order = (
    df_top10_3.groupby('field')['count']
    .sum()
    .sort_values(ascending=False)
    .index
)

# Step 2: Sort df_top10_3 by field (with custom order) and then by count descending
df_top10_3_sorted = (
    df_top10_3
    .assign(field=pd.Categorical(df_top10_3['field'], categories=field_order, ordered=True))
    .sort_values(by=['field', 'count'], ascending=[True, False])
    .reset_index(drop=True)
)

df_top10_3_sorted

,field,all_keywords,count
0,Engineering,"(lithium, renewable energy)",16449
1,Engineering,"(cobalt, renewable energy)",5321
2,Engineering,"(energy demand, lithium)",4734
3,Engineering,"(nickel, renewable energy)",3987
4,Engineering,"(renewable energy, vanadium)",3152
...,...,...,...
383,Psychology,"(energy demand, platinum)",3
384,Psychology,"(renewable energy, uranium)",3
385,Psychology,"(energy demand, graphite, lithium, renewable e...",2
386,Psychology,"(cesium, energy transition)",1


###Consulta de kw para consultar artigos

####Consulta de kw não encontradas

In [ ]:
context = ["energy transition", "climate change"]
problems = []
concepts = [
    "critical mineral",
    "critical metal",
    "critical material",
    "critical raw material",
    "strategic mineral",
    "strategic material",
    "key raw material",
    "rare metal",
    "green transition mineral",
    "green transition metal",
    "green transition material",
    "green transition raw material",
    #"strategic metal",
    "rare earth element",
    "battery metal",
    #"transition metal",
    #"mineral resource",
    "raw material dependenc",
    # "high-demand mineral",
    # "high-demand metal",
    #"material scarci",
    #"resource scarci",
    # "metal sustainab",
    #"mineral sustainab",
    #"resource critical",
    #"material critical",
    "aluminium", "antimony", "arsenic", "barite", "beryllium",
    "bismuth", "cerium", "cesium", "chromium", "cobalt", "dysprosium", "erbium", "europium", "fluorspar",
    "gadolinium", "gallium", "germanium", "graphite", "hafnium", "holmium", "indium", "iridium",
    "lanthanum", "lithium", "lutetium",
    #"magnesium",
    "manganese", "neodymium", "nickel", "niobium",
    "palladium", "platinum", "praseodymium", "rhodium", "rubidium", "ruthenium", "samarium",
    "scandium", "tantalum", "tellurium", "terbium", "thulium", "titanium", "tungsten",
    "uranium", "vanadium", "ytterbium", "yttrium", "zinc", "zirconium", "copper",
    #"clean tech",
    #"critical supply chain",
    #"supply chain vulnerability",
    "rare earth",
    #"battery supply chain",
    "energy storage material"
    #"strategic resource",
    #"low-carbon technology",
    # "raw material governance",
    #"sustainable mining",
    #"circular mining",
    #"secondary sourcing",
    #"geopolitical risk",
    # "critical material recycling",
    #"resource diplomacy",
    #"local content policy",
    #"net-zero strategy",
    #"carbon-neutral technology",
    #"renewable infrastructure"
]

all_found_problems = set([kw.lower() for sublist in work_ids_a_max_fs_3["problem_kw"] for kw in sublist])
all_found_concepts = set([kw.lower() for sublist in work_ids_a_max_fs_3["concept_kw"] for kw in sublist])
all_found_context = set([kw.lower() for sublist in work_ids_a_max_fs_3["context_kw"] for kw in sublist])

# Quais não foram encontradas?
problems_not_found = sorted(set(problems) - all_found_problems)
concepts_not_found = sorted(set(concepts) - all_found_concepts)
context_not_found = sorted(set(context) - all_found_context)

# Mostrar resultados
print("❌ Palavras-chave de 'problems' que não tiveram nenhum match:")
print(problems_not_found)

print("\n❌ Palavras-chave de 'concepts' que não tiveram nenhum match:")
print(concepts_not_found)

print("\n❌ Palavras-chave de 'context' que não tiveram nenhum match:")
print(context_not_found)

❌ Palavras-chave de 'problems' que não tiveram nenhum match:
[]

❌ Palavras-chave de 'concepts' que não tiveram nenhum match:
['fluorspar', 'green transition material', 'green transition metal', 'green transition mineral', 'green transition raw material', 'raw material dependenc']

❌ Palavras-chave de 'context' que não tiveram nenhum match:
['climate change']


In [ ]:
from collections import Counter
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Define the keyword to check
keyword_to_check = "epistemic justice"

# Ensure all_keywords is a list
work_ids_a_max_fs_3["all_keywords"] = work_ids_a_max_fs_3["all_keywords"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Filter rows that include the keyword
filtered = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["all_keywords"].apply(lambda x: keyword_to_check in x)
]

# Convert list of keywords to tuple (for grouping), preserving order (or use sorted(x) if order doesn’t matter)
filtered["keyword_combination"] = filtered["all_keywords"].apply(lambda x: tuple(x))

# Group by field and keyword combination
grouped = (
    filtered.groupby(["field", "keyword_combination", "title", "abstract"])
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

# Show result
grouped


,field,keyword_combination,title,abstract,count


# Group BERTopic filtering

###Setup

In [ ]:
!pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import random

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

work_ids_a_max_fs_3=pd.read_parquet(folder+'group_bronze.parquet')

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning:

invalid escape sequence '\{'



In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

# Step 1 - Embeddings (good baseline; consider multilingual if PT/EN mixed)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2 - UMAP: fewer neighbors -> more local structure (more/smaller clusters)
umap_model = UMAP(
    n_neighbors=15,          # was 15
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# Step 3 - HDBSCAN: split more aggressively
hdbscan_model = HDBSCAN(
    min_cluster_size=5,              # was 15
    min_samples=1,                   # lower = easier to split
    metric="euclidean",              # you can try "cosine" too
    cluster_selection_method="leaf", # finer splits than "eom"
    prediction_data=True
)

# Step 4 - Vectorizer: keep informative terms; allow bigrams
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=3,                        # ignore ultra-rare terms; adjust to corpus size
    stop_words="english"             # add domain stopwords if needed
)

# Step 5 - c-TF-IDF
ctfidf_model = ClassTfidfTransformer()

# Step 6 - Representation model
representation_model = KeyBERTInspired()

# Assemble BERTopic with granularity-friendly options
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    language="english",
    nr_topics=None,                  # do NOT merge topics (keeps granularity)
    low_memory=True,
    verbose=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (3 outputs left out)

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

###Topics

In [ ]:
import pandas as pd

# 1) Prep: one row per (title, abstract) and keep metadata
prep = work_ids_a_max_fs_3.drop_duplicates(subset=['title', 'abstract'])

meta = (
    prep[['id', 'title', 'abstract']]
    .copy()
)

# Drop rows with nulls or empty/whitespace-only title/abstract
meta[['title', 'abstract']] = meta[['title', 'abstract']].astype('string')
meta = (
    meta
    .replace({'title': r'^\s*$', 'abstract': r'^\s*$'}, pd.NA, regex=True)
    .dropna(subset=['id', 'title', 'abstract'])
    .reset_index(drop=True)
)


In [ ]:
len(meta)

8527

In [ ]:
#meta = meta.iloc[random.sample(range(len(meta)), min(4000, len(meta)))]

In [ ]:
# 2) Build docs (list[str]) for BERTopic from title + abstract
meta['doc'] = (
    meta[['title', 'abstract']]
    .agg('. '.join, axis=1)               # join with period + space
    .str.replace(r'\s+', ' ', regex=True) # collapse whitespace
    .str.strip()
)

# Keep only non-empty docs and preserve alignment
mask = meta['doc'].ne('')
meta = meta.loc[mask].reset_index(drop=True)
docs = meta['doc'].tolist()


In [ ]:
# 3) Fit / transform
topics, probs = topic_model.fit_transform(docs)

2025-11-02 15:08:00,414 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/267 [00:00<?, ?it/s]

2025-11-02 15:20:57,034 - BERTopic - Embedding - Completed ✓
2025-11-02 15:20:57,035 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-11-02 15:21:25,698 - BERTopic - Dimensionality - Completed ✓
2025-11-02 15:21:25,700 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-11-02 15:21:25,983 - BERTopic - Cluster - Completed ✓
2025-11-02 15:21:25,992 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-11-02 15:23:20,525 - BERTopic - Representation - Completed ✓


In [ ]:
# Mantém o gráfico base
fig_barchart = topic_model.visualize_barchart(
    width=280,
    height=330,
    top_n_topics=20,
    n_words=10
)

# --- Ajustes de estilo ABNT/DPCT ---
fig_barchart.update_traces(
    marker_color="gray",         # cor única
    marker_line_color="black",   # contorno preto fino
    marker_line_width=0.5,       # espessura discreta
    opacity=0.85                 # leve transparência
)

fig_barchart.update_layout(
    title_text="Tópicos Encontrados",
    font=dict(family="Times New Roman", size=12),
    title=dict(font=dict(size=14)),
    plot_bgcolor="white",        # fundo branco
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=20, t=60, b=60)
)

# renomeia os títulos de cada subplot
for i, ann in enumerate(fig_barchart.layout.annotations):
    ann.text = f"Tópico {i+1}"
    ann.font = dict(family="Times New Roman", size=11)

fig_barchart.show()


In [ ]:
document_info = topic_model.get_document_info(docs).copy()  # expects the same list[str]
document_info['id'] = meta['id'].to_numpy()                 # align by position

In [ ]:
neg_topic=document_info[document_info['Topic'] == -1]
#neg_topic[['Topic', 'Name','Top_n_words','Probability','Representative_document','id']]
neg_topic['Top_n_words'].unique()

array(['renewable energy - renewable - solar energy - energy storage - photovoltaics - photovoltaic - solar cells - energy materials - lithium ion - hydrogen'],
      dtype=object)

In [ ]:
ids=neg_topic['id'].unique()
ids=pd.DataFrame(ids, columns = ['id'])
ids

,id
0,2085495366
1,2134019247
2,2084674899
3,2249485850
4,2166515142
...,...
2779,4378528461
2780,4320486981
2781,4392704861
2782,4385597273


In [ ]:
filtered = work_ids_a_max_fs_3[work_ids_a_max_fs_3['id'].isin(ids['id'])]
filtered[['id','title','abstract']].drop_duplicates(subset=['id'])

,id,title,abstract
17,2085495366,"Trans10, cis12-conjugated linoleic acid preven...",A group of polyunsaturated fatty acids called ...
32,2134019247,Comparison of aluminium and stainless steel bu...,This article provides the results and findings...
38,2084674899,Impact of Steam on the Interaction between Bio...,The combination of solid oxide fuel cells (SOF...
50,2249485850,Nuclear Offers Up Explosive Debate,Panel session held at Paydirt's Uranium Confer...
124,2166515142,MHTGR/coal gasification-an environmentally acc...,"A new generation of nuclear power plants, such..."
...,...,...,...
186020,4378528461,Research of an Abandoned Tailings Deposit in t...,Global situations such as economic recovery af...
186071,4320486981,Our Next Energy gets cash for battery factory,The battery company Our Next Energy (ONE) has ...
186387,4392704861,Innovating Flexible Lithium-Ion Battery Design...,Innovating Flexible Lithium-Ion Battery Design...
186429,4385597273,Optical Properties and Phase Transition in VO<...,Energy saving methods/materials are becoming i...


In [ ]:
# 4) Get per-document info and reattach IDs
document_info = topic_model.get_document_info(docs).copy()  # expects the same list[str]
document_info['id'] = meta['id'].to_numpy()                 # align by position

In [ ]:
# 5) Retrieve all IDs with Topic > 0  (exclude outliers -1 and topic 0)
ids_topic_gt0 = document_info.loc[document_info['Topic'] >= 0, 'id'].tolist()

In [ ]:
len(ids_topic_gt0)

5743

###Saves topics

In [ ]:
topic_model.save(folder+'group_topics')

2025-11-02 15:23:26,163 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
#Salva os tópicos únicos e suas informações
unique_topics=document_info['Topic'].unique()
unique_topics=document_info[document_info['Topic'].isin(unique_topics)]
unique_topics=unique_topics[['id','Topic','Name','Top_n_words']].drop_duplicates(subset=['id','Topic','Top_n_words'])
df=unique_topics.sort_values(by='Topic')
df.to_parquet(folder + 'group_bronze_topics.parquet', index=False)

#Group topics shortcut

###Visualização dos tópicos

In [ ]:
from bertopic import BERTopic
import os

topic_model = BERTopic.load(os.path.join(folder, 'group_topics'))

ModuleNotFoundError: No module named 'bertopic'

In [ ]:
# Mantém o gráfico base
fig_barchart = topic_model.visualize_barchart(
    width=280,
    height=330,
    top_n_topics=20,
    n_words=10
)

# --- Ajustes de estilo ABNT/DPCT ---
fig_barchart.update_traces(
    marker_color="gray",         # cor única
    marker_line_color="black",   # contorno preto fino
    marker_line_width=0.5,       # espessura discreta
    opacity=0.85                 # leve transparência
)

fig_barchart.update_layout(
    title_text="Tópicos Encontrados",
    font=dict(family="Times New Roman", size=12),
    title=dict(font=dict(size=14)),
    plot_bgcolor="white",        # fundo branco
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=20, t=60, b=60)
)

# renomeia os títulos de cada subplot
for i, ann in enumerate(fig_barchart.layout.annotations):
    ann.text = f"Tópico {i+1}"
    ann.font = dict(family="Times New Roman", size=11)

fig_barchart.show()


##Distribuição de tópicos

In [ ]:
import pandas as pd
df=pd.read_parquet(folder + 'group_bronze_topics.parquet')
df

In [ ]:
len(df.drop_duplicates(subset=['id']))

In [ ]:
import pandas as pd

# Group by topic name and top words, counting how many articles per group
topic_summary = (
    df[df['Topic'] > -1]  # filter out the outlier topic (-1)
    .groupby(['Topic', 'Name', 'Top_n_words'])
    .agg(article_count=('id', 'count'))
    .reset_index()
    .sort_values('article_count', ascending=False)
)

topic_summary


In [ ]:
import plotly.express as px

# Sort by article_count (descending) and keep top 30 for readability
topic_summary_top30 = topic_summary.sort_values('article_count', ascending=False).head(30)

# Create bar chart (monochrome / ABNT style)
fig = px.bar(
    topic_summary_top30,
    x='Name',
    y='article_count',
    text='article_count',
    title='Distribuição de artigos por tópico (Top 30)',
)

# Style: grayscale, Times New Roman, axis lines
fig.update_traces(
    textposition='outside',
    marker_color='rgba(80,80,80,1)',
    marker_line_color='black',
    marker_line_width=0.8
)

fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=600,
    margin=dict(l=60, r=40, t=80, b=140),
    xaxis=dict(
        title='Tópico',
        tickangle=45,
        linecolor='black',
        mirror=True,
        showgrid=False,
    ),
    yaxis=dict(
        title='Número de artigos',
        tickmode='linear',
        dtick=5,  # mínimo espaçamento de 5 entre ticks
        linecolor='black',
        mirror=True,
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
    ),
    showlegend=False,
)

fig.show()


NameError: name 'topic_summary' is not defined

##IDs para análise

In [ ]:
ids_df = pd.DataFrame(ids_topic_gt0, columns=['id'])
len(ids_df)

In [ ]:
ids_df.to_parquet(folder + 'group_gold.parquet', index=False)

#[SHORTCUT GROUP GOLD]

In [ ]:
import pandas as pd

folder = '/content/drive/MyDrive/cientometria_resultados/Resultados preliminares/'
work_ids_a_max_fs_3 = pd.read_parquet(folder + 'group_bronze.parquet')
ids_df=pd.read_parquet(folder + 'group_gold.parquet')

In [ ]:
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

8668

In [ ]:
len(ids_df.drop_duplicates(subset=['id']))

5743

##Análise de publicações

###Publicações por área do conhecimento

In [ ]:
ids_set = set(ids_df['id']) #loads all IDs that belong to a topic >=0
work_ids_a_max_fs_3 = work_ids_a_max_fs_3[work_ids_a_max_fs_3['id'].isin(ids_set)].copy() #filters IDs that belong to a topic >=0


len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

5743

In [ ]:
df=work_ids_a_max_fs_3
df_filtrado = df[df["all_keywords"].str.contains("energy transition", case=False, na=False)]

,id,doi,title,type,publication_year,max_score,abstract,topic,field,subfield,...,institution_id,author_id,author,funder_id,funder,institution,problem_kw,concept_kw,context_kw,all_keywords


In [ ]:
area_do_conhecimento=work_ids_a_max_fs_3.drop_duplicates(subset=['id','field'])
area_do_conhecimento[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
8,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,"[cobalt, energy supply]"
9,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,G-Quadruplex DNA Structures and Functions,"Biochemistry, Genetics and Molecular Biology",Molecular Biology,Japan,"[cobalt, energy supply]"
16,2028504728,10.5070/g311910544,The Solar Economy: Renewable Energy for a Sustainable Global Future,2003,0.7278,Global Energy Transition and Fossil Fuel Depletion,Energy,"Renewable Energy, Sustainability and the Environment",None,"[renewable energy, uranium]"
23,60233774,None,"Multiple criteria deciding on phytoremediation of a heavy metal contaminated agricultural area. Case: The Campine, Belgium",2008,0.9566,Urban Metabolism and Sustainability Assessment,Environmental Science,Environmental Engineering,Belgium,"[renewable energy, zinc]"
24,60233774,None,"Multiple criteria deciding on phytoremediation of a heavy metal contaminated agricultural area. Case: The Campine, Belgium",2008,0.9566,Development and Impacts of Bioenergy Crops,Agricultural and Biological Sciences,Agronomy and Crop Science,Belgium,"[renewable energy, zinc]"


In [ ]:
area_do_conhecimento[
    area_do_conhecimento['field'] == "Engineering"
][['id', 'doi', 'title', 'publication_year', 'max_score', 'topic', 'field', 'subfield', 'country', 'all_keywords']]


,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
31,2118211766,10.1109/tie.2009.2036020,Search for an Optimal Rapid-Charging Pattern for Li-Ion Batteries Using the Taguchi Approach,2009,0.9997,Lithium-ion Battery Management in Electric Vehicles,Engineering,Automotive Engineering,Taiwan,"[lithium, renewable energy]"
52,2121308869,10.1021/cm803259x,Ionothermal Synthesis of Tailor-Made LiFePO<sub>4</sub> Powders for Li-Ion Battery Applications,2009,1.0000,Battery Recycling and Rare Earth Recovery,Engineering,Mechanical Engineering,France,"[energy demand, lithium]"
81,1987375721,10.1063/1.1989431,Electrochemically controlled transport of lithium through ultrathin SiO2,2005,1.0000,Lithium-ion Battery Technology,Engineering,Electrical and Electronic Engineering,United States,"[cobalt, energy supply, lithium]"
141,2467919703,None,Bio-Battery Research or Cow Dung Battery Research,2009,0.6981,Origins and Applications of Electric Circuits,Engineering,Electrical and Electronic Engineering,None,"[renewable energy, zinc]"
142,1995081886,10.3844/ajassp.2009.332.336,Promotion of Coconut Shell Gasification by Steam Reforming on Nickel-Dolomite,2009,0.9997,Biomass Pyrolysis and Conversion Technologies,Engineering,Biomedical Engineering,Thailand,"[nickel, renewable energy]"
...,...,...,...,...,...,...,...,...,...,...
186439,4386592576,10.1002/anie.202311032,Lithium Bis(oxalate)borate Additive for Self‐repairing Zincophilic Solid Electrolyte Interphases towards Ultrahigh‐rate and Ultra‐stable Zinc Anodes,2023,1.0000,Aqueous Zinc-Ion Battery Technology,Engineering,Electrical and Electronic Engineering,China,"[lithium, renewable energy, zinc]"
186539,4386883494,10.1109/sefet57834.2023.10244838,Design of an Air-Based Thermal Management and Predictions of Thermal Characteristics of Lithium-Ion Cell Using 1D Numerical Model,2023,1.0000,Lithium Battery Technologies,Engineering,Electrical and Electronic Engineering,India,"[lithium, renewable energy]"
186545,4386879271,10.3390/sym15091794,Al2O3-Cu\\Ethylene Glycol-Based Magnetohydrodynamic Non-Newtonian Maxwell Hybrid Nanofluid Flow with Suction Effects in a Porous Space: Energy Saving by Solar Radiation,2023,1.0000,Solar Air Heater Heat Transfer Analysis,Engineering,Mechanical Engineering,Saudi Arabia,"[copper, renewable energy]"
186626,4383041355,10.1080/10407782.2023.2229011,Entropy analysis of unsteady MHD nanofluid flow over a stretching surface with effects of variable viscosity and nonuniform heat generation,2023,1.0000,Turbulent Flows and Vortex Dynamics,Engineering,Computational Mechanics,Nigeria,"[copper, renewable energy, titanium]"


In [ ]:
len(area_do_conhecimento)

10643

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ===== Dados =====
count_by_field = (
    area_do_conhecimento['field']
    .value_counts()
    .reset_index()
)
count_by_field.columns = ['field', 'count']
count_by_field = count_by_field.sort_values('count', ascending=False)

total = count_by_field['count'].sum()
count_by_field['cumulative_percentage'] = (
    count_by_field['count'].cumsum() / total * 100
)

# ===== Figura (ABNT/DPCT) =====
FIG_NUM   = 1
TITULO    = "Distribuição de artigos por área do conhecimento e porcentagem cumulativa"

fig = go.Figure()

# Barras (número de publicações) — cinza
fig.add_trace(
    go.Bar(
        x=count_by_field['field'],
        y=count_by_field['count'],
        name='Número de publicações',
        marker_color='rgba(0,0,0,0.65)',
        yaxis='y1'
    )
)

# Linha (porcentagem cumulativa) — preto tracejado, marcadores brancos com borda preta
fig.add_trace(
    go.Scatter(
        x=count_by_field['field'],
        y=count_by_field['cumulative_percentage'],
        name='Porcentagem cumulativa',
        mode='lines+markers+text',
        line=dict(color='black', width=2, dash='dash'),
        marker=dict(size=6, color='white', line=dict(width=1, color='black')),
        text=[f"{int(round(p))}%" for p in count_by_field['cumulative_percentage']],
        textposition='top center',
        yaxis='y2'
    )
)

# Layout ABNT/DPCT – título acima, tipografia Times New Roman, eixos e grades discretas
fig.update_layout(
    title=f"{TITULO}",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    height=520,
    margin=dict(l=70, r=70, t=80, b=100),  # margem inferior suficiente mesmo sem 'Fonte'
    xaxis=dict(
        title='Área do conhecimento',
        tickangle=-45,
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis2=dict(
        title='Porcentagem cumulativa (%)',
        rangemode='tozero',
        overlaying='y',
        side='right',
        range=[0, 110],
        showgrid=False,
        linecolor='black'
    )
)

fig.show()

# Exportar em alta resolução (opcional para inserir na dissertação)
# fig.write_image("figura_1_pareto_fields.pdf")   # requer 'kaleido' instalado
# fig.write_image("figura_1_pareto_fields.png", scale=3)


####Publicações por área (df)

In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = area_do_conhecimento['field'].value_counts().reset_index()
count_by_field.columns = ['Área do conhecimento', 'Contagem']

# Exibir o DataFrame
count_by_field

,Área do conhecimento,Contagem
0,Engineering,4374
1,Energy,2022
2,Materials Science,1770
3,Environmental Science,716
4,Chemistry,442
5,Chemical Engineering,367
6,Physics and Astronomy,185
7,Computer Science,148
8,Earth and Planetary Sciences,105
9,"Biochemistry, Genetics and Molecular Biology",100


In [ ]:
count_by_field['Contagem'].sum()

np.int64(10643)

###Distribuição de publicação por área ao longo do tempo

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# 1) Aggregate data
publications_per_year_field = (
    area_do_conhecimento
    .groupby(['publication_year','field'])
    .size()
    .reset_index(name='count')
)

# 2) Pivot to cumulative counts
pivot_df = (
    publications_per_year_field
    .pivot_table(index='publication_year', columns='field', values='count', aggfunc='sum', fill_value=0)
    .sort_index()
    .cumsum()
)

# 3) Order columns by final total
pivot_df = pivot_df[pivot_df.iloc[-1].sort_values(ascending=False).index]

# 4) Build combined style list
marker_symbols = ['circle','square','diamond','triangle-up','triangle-down',
                  'triangle-left','triangle-right','cross','x','star']
dash_styles    = ['solid','dash','dot','longdash','dashdot']

# a) dash + marker combos
dash_marker_styles = [(m, d, 'lines+markers') for d in dash_styles for m in marker_symbols]
# b) dash only
dash_only_styles   = [(None, d, 'lines') for d in dash_styles]
# c) marker only
marker_only_styles = [(m, None, 'markers') for m in marker_symbols]

# combine all (you can reorder if you want more of some category)
style_pool = dash_marker_styles + dash_only_styles + marker_only_styles
styles = cycle(style_pool)

# 5) Build figure
fig = go.Figure()
first_year = int(pivot_df.index.min())
last_year  = int(pivot_df.index.max())

for col in pivot_df.columns:
    marker_sym, dash, mode = next(styles)

    # base marker dict
    marker_dict = dict(size=7, color='white', line=dict(color='black', width=1))
    if marker_sym:
        marker_dict['symbol'] = marker_sym

    # base line dict
    line_dict = dict(color='black', width=1.8)
    if dash:
        line_dict['dash'] = dash

    fig.add_trace(
        go.Scatter(
            x=pivot_df.index.astype(int),
            y=pivot_df[col],
            mode=mode,
            name=col,
            line=line_dict,
            marker=marker_dict,
            hovertemplate=f"Ano: %{{x}}<br>Acumulado: %{{y}}<extra>{col}</extra>"
        )
    )

# 6) Layout ABNT/DPCT
fig.update_layout(
    title="Volume acumulado de publicações por ano, por área do conhecimento",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(
        title="Ano de publicação",
        type='linear',
        range=[first_year, last_year],
        dtick=1,
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Número acumulado de publicações",
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# 1) Aggregate data
publications_per_year_field = (
    area_do_conhecimento
    .groupby(['publication_year', 'field'])
    .size()
    .reset_index(name='count')
)

# 2) Pivot: cumulative counts
pivot_df = (
    publications_per_year_field
    .pivot_table(index='publication_year', columns='field',
                 values='count', aggfunc='sum', fill_value=0)
    .sort_index()
    .cumsum()
)

# 3) Order columns by final total
pivot_df = pivot_df[pivot_df.iloc[-1].sort_values(ascending=False).index]

# 4) Build combined style list
marker_symbols = ['circle','square','diamond','triangle-up','triangle-down',
                  'triangle-left','triangle-right','cross','x','star']
dash_styles    = ['solid','dash','dot','longdash','dashdot']

# a) dash + marker combos
dash_marker_styles = [(m, d, 'lines+markers') for d in dash_styles for m in marker_symbols]
# b) dash only
dash_only_styles   = [(None, d, 'lines') for d in dash_styles]
# c) marker only
marker_only_styles = [(m, None, 'markers') for m in marker_symbols]

# Combine for a large style pool
style_pool = dash_marker_styles + dash_only_styles + marker_only_styles
styles = cycle(style_pool)

# 5) Figure (stacked percent)
fig = go.Figure()
first_year = int(pivot_df.index.min())
last_year  = int(pivot_df.index.max())

for col in pivot_df.columns:
    marker_sym, dash, mode = next(styles)

    # build marker and line dicts
    marker_dict = dict(size=7, color='white', line=dict(color='black', width=1))
    if marker_sym:
        marker_dict['symbol'] = marker_sym

    line_dict = dict(color='black', width=1.5)
    if dash:
        line_dict['dash'] = dash

    fig.add_trace(
        go.Scatter(
            x=pivot_df.index.astype(int),
            y=pivot_df[col],
            mode=mode,
            name=col,
            stackgroup='one',          # stack to 100 %
            groupnorm='percent',
            fill='none',               # keep pure white background
            line=line_dict,
            marker=marker_dict,
            hovertemplate=f"Ano: %{{x}}<br>% da área: %{{y:.1f}}%<extra>{col}</extra>"
        )
    )

# 6) Layout ABNT/DPCT
fig.update_layout(
    title="Proporção acumulada de publicações por ano, por área do conhecimento",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(
        title="Ano de publicação",
        type='linear',                   # numeric axis
        range=[first_year, last_year],   # start exactly at first year
        dtick=1,                          # one tick per year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title="Percentual (%)",
        range=[0, 100],
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()

###Publicações por país

In [ ]:
paises=work_ids_a_max_fs_3.drop_duplicates(subset=['id','country'])
paises[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
8,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,"[cobalt, energy supply]"
16,2028504728,10.5070/g311910544,The Solar Economy: Renewable Energy for a Sustainable Global Future,2003,0.7278,Global Energy Transition and Fossil Fuel Depletion,Energy,"Renewable Energy, Sustainability and the Environment",None,"[renewable energy, uranium]"
23,60233774,None,"Multiple criteria deciding on phytoremediation of a heavy metal contaminated agricultural area. Case: The Campine, Belgium",2008,0.9566,Urban Metabolism and Sustainability Assessment,Environmental Science,Environmental Engineering,Belgium,"[renewable energy, zinc]"
29,2118211766,10.1109/tie.2009.2036020,Search for an Optimal Rapid-Charging Pattern for Li-Ion Batteries Using the Taguchi Approach,2009,0.9997,Experimental Design and Optimization Methods,Decision Sciences,Management Science and Operations Research,Taiwan,"[lithium, renewable energy]"
49,1652633633,None,"Earthship Fife construction workshop run by Michael Reynolds [participation, research] Kinghorn, Scotland; 1-10 Aug 2002",2002,0.5957,Impact of Information Technology Infrastructure on Various Sectors,Computer Science,Information Systems,None,"[aluminium, renewable energy]"


In [ ]:
len(paises)

7329

In [ ]:
import plotly.graph_objects as go

# Contar ocorrência de cada país
count_by_country = paises['country'].value_counts().reset_index()
count_by_country.columns = ['country', 'count']
count_by_country = count_by_country.sort_values('count', ascending=False)

# Porcentagem cumulativa
total = count_by_country['count'].sum()
count_by_country['cumulative_percentage'] = (
    count_by_country['count'].cumsum() / total * 100
)

# Top 20
top20 = count_by_country.head(20)

# Figura
fig = go.Figure()

# Barras — cinza
fig.add_trace(
    go.Bar(
        x=top20['country'],
        y=top20['count'],
        name='Número de publicações',
        marker_color='rgba(0,0,0,0.65)',  # cinza escuro
        yaxis='y1'
    )
)

# Linha — preta tracejada, marcadores brancos com borda preta
fig.add_trace(
    go.Scatter(
        x=top20['country'],
        y=top20['cumulative_percentage'],
        name='Porcentagem cumulativa',
        mode='lines+markers+text',
        line=dict(color='black', width=2, dash='dash'),
        marker=dict(size=6, color='white', line=dict(width=1, color='black')),
        text=[f"{int(p)}%" for p in top20['cumulative_percentage']],
        textposition='top center',
        yaxis='y2'
    )
)

# Layout ABNT/DPCT
fig.update_layout(
    title='Top 20 países por número de publicações e porcentagem cumulativa',
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    height=560,
    margin=dict(l=80, r=80, t=80, b=120),
    xaxis=dict(
        title='País',
        tickangle=-45,
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis2=dict(
        title='Porcentagem cumulativa (%)',
        rangemode='tozero',
        overlaying='y',
        side='right',
        range=[0, 110],
        showgrid=False,
        linecolor='black'
    ),
    bargap=0.2
)

fig.show()


In [ ]:
count_by_country


,country,count,cumulative_percentage
0,United States,1152,18.408437
1,China,857,32.102908
2,Germany,369,37.999361
3,India,303,42.841163
4,United Kingdom,277,47.267498
...,...,...,...
115,Greenland,1,99.936082
116,Democratic Republic of the Congo,1,99.952061
117,Moldova,1,99.968041
118,Cameroon,1,99.984020


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = paises['field'].value_counts().reset_index()
count_by_field.columns = ['Área do conhecimento', 'Contagem']

# Exibir o DataFrame
count_by_field

,Área do conhecimento,Contagem
0,Engineering,3649
1,Energy,1206
2,Materials Science,1083
3,Environmental Science,445
4,Chemistry,211
5,Chemical Engineering,158
6,Physics and Astronomy,119
7,Computer Science,90
8,"Biochemistry, Genetics and Molecular Biology",69
9,Social Sciences,63


In [ ]:
count_by_field['Contagem'].sum()

np.int64(7329)

###Publicações por ano

In [ ]:
ano=work_ids_a_max_fs_3.drop_duplicates(subset=['id'])
ano[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
8,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,"[cobalt, energy supply]"
16,2028504728,10.5070/g311910544,The Solar Economy: Renewable Energy for a Sustainable Global Future,2003,0.7278,Global Energy Transition and Fossil Fuel Depletion,Energy,"Renewable Energy, Sustainability and the Environment",None,"[renewable energy, uranium]"
23,60233774,None,"Multiple criteria deciding on phytoremediation of a heavy metal contaminated agricultural area. Case: The Campine, Belgium",2008,0.9566,Urban Metabolism and Sustainability Assessment,Environmental Science,Environmental Engineering,Belgium,"[renewable energy, zinc]"
29,2118211766,10.1109/tie.2009.2036020,Search for an Optimal Rapid-Charging Pattern for Li-Ion Batteries Using the Taguchi Approach,2009,0.9997,Experimental Design and Optimization Methods,Decision Sciences,Management Science and Operations Research,Taiwan,"[lithium, renewable energy]"
49,1652633633,None,"Earthship Fife construction workshop run by Michael Reynolds [participation, research] Kinghorn, Scotland; 1-10 Aug 2002",2002,0.5957,Impact of Information Technology Infrastructure on Various Sectors,Computer Science,Information Systems,None,"[aluminium, renewable energy]"


In [ ]:
len(ano)

5743

In [ ]:
import pandas as pd
import plotly.express as px

# ===================== 1) Data =====================
publications_per_year = (
    ano['publication_year']
    .value_counts()
    .reset_index()
)
publications_per_year.columns = ['publication_year', 'count']
publications_per_year = publications_per_year.sort_values('publication_year')

# Limitar até 2023
publications_per_year = publications_per_year[
    publications_per_year['publication_year'] <= 2023
]

# make sure the year is numeric
publications_per_year['publication_year'] = publications_per_year['publication_year'].astype(int)

first_year = int(publications_per_year['publication_year'].min())
last_year  = int(publications_per_year['publication_year'].max())

# ===================== 2) Figure =====================
fig = px.line(
    publications_per_year,
    x='publication_year',
    y='count',
    markers=True,
    title='Volume de publicações por ano',
    labels={
        'publication_year': 'Ano de publicação',
        'count': 'Número de publicações'
    }
)

fig.update_traces(
    line=dict(color='black', width=2),
    marker=dict(size=6, color='white', line=dict(width=1, color='black'))
)

# ===================== 3) Layout =====================
fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    height=560,
    margin=dict(l=80, r=80, t=80, b=80),
    xaxis=dict(
        title='Ano de publicação',
        type='linear',                      # treat as numeric
        range=[first_year, last_year],      # start at first year
        dtick=1,                             # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    )
)

fig.show()


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_year = ano['publication_year'].value_counts().reset_index()

# Exibir o DataFrame
count_by_year

,publication_year,count
0,2023,1006
1,2022,768
2,2020,532
3,2021,531
4,2019,469
5,2018,409
6,2016,372
7,2017,353
8,2015,305
9,2014,249


In [ ]:
count_by_year['count'].sum()

np.int64(5743)

###Publicações por país/ano

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# ===== 1) Dados =====
publications_per_year_country = (
    paises
    .groupby(['publication_year', 'country'])
    .size()
    .reset_index(name='count')
)

# Pivot (anos x países) e cumulativo
pivot_df = (
    publications_per_year_country
    .pivot_table(index='publication_year', columns='country', values='count', aggfunc='sum', fill_value=0)
    .sort_index()
    .cumsum()
)

# Selecionar Top 20 países pelo total acumulado
top_countries = pivot_df.iloc[-1].sort_values(ascending=False).head(5).index
pivot_df = pivot_df[top_countries]

# ===== 2) Estilos =====
marker_symbols = ['circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
                  'triangle-left', 'triangle-right', 'cross', 'x', 'star']
dash_styles    = ['solid', 'dash']
styles = cycle([(m, d) for d in dash_styles for m in marker_symbols])

# ===== 3) Figura Plotly =====
fig = go.Figure()

for col in pivot_df.columns:
    marker_sym, dash = next(styles)
    fig.add_trace(
        go.Scatter(
            x=pivot_df.index,   # keep as numeric for a true numeric axis
            y=pivot_df[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.5, dash=dash),
            marker=dict(symbol=marker_sym, size=6,
                        color='white', line=dict(color='black', width=1)),
            hovertemplate="Ano: %{x}<br>Acumulado: %{y}<extra>"+str(col)+"</extra>"
        )
    )

# ===== 4) Layout ABNT/DPCT-IG =====
first_year = int(pivot_df.index.min())
last_year  = int(pivot_df.index.max())

fig.update_layout(
    title="Volume acumulado de publicações por ano – Top 5 países",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    xaxis=dict(
        title="Ano de publicação",
        type='linear',                # continuous numeric axis
        range=[first_year, last_year],# start exactly at the first available year
        dtick=1,                      # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title="Número acumulado de publicações",
        rangemode='tozero',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
pivot_df

country,United States,China,Germany,India,United Kingdom
publication_year,,,,,
2000,0,1,2,0,0
2001,3,1,2,0,0
2002,7,1,4,1,0
2003,7,2,4,1,0
2004,13,2,4,1,0
2005,17,4,5,1,1
2006,27,5,5,1,1
2007,35,9,5,1,1
2008,42,16,8,2,1


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from itertools import cycle

# ==============================================
# 1) Prepare data
# ==============================================
publications_per_year_country = (
    paises
    .groupby(['publication_year', 'country'])
    .size()
    .reset_index(name='count')
)

pivot_df = publications_per_year_country.pivot_table(
    index='publication_year',
    columns='country',
    values='count',
    aggfunc='sum',
    fill_value=0
)

# Keep top 20 contributors
top_countries = pivot_df.sum().nlargest(20).index
pivot_df = pivot_df[top_countries].sort_index().cumsum()

# Percent of total per year (stacked 100 %)
pivot_df_percent = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

# Order by final contribution
country_order = pivot_df.iloc[-1].sort_values(ascending=False).index.tolist()
pivot_df_percent = pivot_df_percent[country_order]

# Make sure year is numeric and sorted
pivot_df_percent.index = pivot_df_percent.index.astype(int)
first_year = int(pivot_df_percent.index.min())
last_year  = int(pivot_df_percent.index.max())

# ==============================================
# 2) Define line+marker styles
# ==============================================
marker_symbols = ['circle','square','diamond','triangle-up','triangle-down',
                  'triangle-left','triangle-right','cross','x','star']
dash_styles    = ['solid','dash']
styles = cycle([(m, d) for d in dash_styles for m in marker_symbols])

# ==============================================
# 3) Build stacked 100 % line/area chart
# ==============================================
fig = go.Figure()

for col in pivot_df_percent.columns:
    marker_sym, dash = next(styles)
    fig.add_trace(
        go.Scatter(
            x=pivot_df_percent.index,
            y=pivot_df_percent[col],
            mode='lines+markers',
            name=col,
            stackgroup='one',           # stack to 100 %
            groupnorm='percent',
            fill='none',                # transparent fill
            line=dict(color='black', width=1.5, dash=dash),
            marker=dict(symbol=marker_sym, size=6,
                        color='white', line=dict(color='black', width=1)),
            hoverinfo='skip'            # <<< no hover boxes
        )
    )

# ==============================================
# 4) Layout in ABNT/DPCT style
# ==============================================
fig.update_layout(
    title="Proporção acumulada de publicações por ano – Top 20 países",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    paper_bgcolor='white',   # outer background
    plot_bgcolor='white',    # plotting area background
    xaxis=dict(
        title='Ano de publicação',
        type='linear',                   # numeric axis
        range=[first_year, last_year],   # start at first actual year
        dtick=1,                         # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Percentual acumulado (%)',
        range=[0, 100],
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.25,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = work_ids_a_max_fs_3['publication_year'].value_counts().reset_index()
count_by_field.columns = ['Ano de publicação', 'Contagem']

# Exibir o DataFrame
print(count_by_field)

    Ano de publicação  Contagem
0                2023     25535
1                2022     18738
2                2021     15431
3                2020     13839
4                2019     11617
5                2018      9755
6                2017      6505
7                2016      5808
8                2015      4682
9                2014      2593
10               2013      1564
11               2012       965
12               2011       958
13               2009       594
14               2010       570
15               2007       352
16               2008       300
17               2006       244
18               2000       199
19               2005       189
20               2002       132
21               2004       101
22               2003        81
23               2001        38


In [ ]:
count_by_field['Contagem'].sum()

np.int64(120790)

###Publicações por instituição

In [ ]:
institution=work_ids_a_max_fs_3.drop_duplicates(subset=['id','institution'])
institution[['id','doi','title','publication_year','max_score','topic','field','subfield','country','author','institution','funder','all_keywords']].head(5)

,id,doi,title,publication_year,max_score,topic,field,subfield,country,author,institution,funder,all_keywords
8,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,Yuzo Yoshikawa,Okayama University,None,"[cobalt, energy supply]"
16,2028504728,10.5070/g311910544,The Solar Economy: Renewable Energy for a Sustainable Global Future,2003,0.7278,Global Energy Transition and Fossil Fuel Depletion,Energy,"Renewable Energy, Sustainability and the Environment",None,None,None,None,"[renewable energy, uranium]"
23,60233774,None,"Multiple criteria deciding on phytoremediation of a heavy metal contaminated agricultural area. Case: The Campine, Belgium",2008,0.9566,Urban Metabolism and Sustainability Assessment,Environmental Science,Environmental Engineering,Belgium,Nele Witters,Hasselt University,None,"[renewable energy, zinc]"
29,2118211766,10.1109/tie.2009.2036020,Search for an Optimal Rapid-Charging Pattern for Li-Ion Batteries Using the Taguchi Approach,2009,0.9997,Experimental Design and Optimization Methods,Decision Sciences,Management Science and Operations Research,Taiwan,Yihua Liu,National Taiwan University of Science and Technology,None,"[lithium, renewable energy]"
49,1652633633,None,"Earthship Fife construction workshop run by Michael Reynolds [participation, research] Kinghorn, Scotland; 1-10 Aug 2002",2002,0.5957,Impact of Information Technology Infrastructure on Various Sectors,Computer Science,Information Systems,None,None,None,None,"[aluminium, renewable energy]"


In [ ]:
len(institution)

10341

In [ ]:
institution_count=institution['institution'].value_counts().reset_index()
institution_count.columns = ['institution', 'count']
institution_count = institution_count.sort_values(by='count', ascending=False)
institution_count

,institution,count
0,Chinese Academy of Sciences,103
1,Argonne National Laboratory,75
2,Oak Ridge National Laboratory,73
3,Tsinghua University,58
4,National Renewable Energy Laboratory,51
...,...,...
2029,Matís (Iceland),1
1998,Institut de Recherche en Informatique et Systèmes Aléatoires,1
1999,Central Institute of Plastics Engineering and Technology,1
2000,Université Hassan II Mohammedia,1


In [ ]:
import plotly.express as px
import numpy as np

TOP_N = 20
plot_df = institution_count.head(TOP_N)

# paleta em tons de cinza (do mais escuro para o mais claro)
grays = [f"rgb({v},{v},{v})" for v in np.linspace(30, 200, num=TOP_N, dtype=int)]

fig = px.bar(
    plot_df,
    x="count",
    y="institution",
    orientation="h",
    text="count",
    labels={"count": "Número de publicações", "institution": "Instituição"},
    title=f"Top {TOP_N} instituições por número de publicações",
    color_discrete_sequence=grays
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker_line_width=1,
    marker_line_color="black"
)

fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    xaxis=dict(
        title='Número de publicações',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Instituição',
        categoryorder='total ascending',  # maior valor no topo
        showgrid=False,
        linecolor='black', mirror=True
    ),
    margin=dict(l=220, r=40, t=80, b=60),  # espaço extra para nomes longos
    height=700
)

fig.show()


##WIP

In [ ]:
prep=work_ids_a_max_fs_3[['id', 'doi','title','field','abstract','subfield']].drop_duplicates(subset='id')

final = prep[prep['field'].isin(['Engineering','Energy','Materials Science'])]
final.to_csv(folder+'final.csv', index=False)

In [ ]:
df=work_ids_a_max_fs_3
df[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
8,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,"[cobalt, energy supply]"
9,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,G-Quadruplex DNA Structures and Functions,"Biochemistry, Genetics and Molecular Biology",Molecular Biology,Japan,"[cobalt, energy supply]"
10,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Self-Assembly and Molecular Recognition in Chemistry,Chemistry,Organic Chemistry,Japan,"[cobalt, energy supply]"
13,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,Fluorescent Chemosensors for Ion Detection and Bioimaging,Chemistry,Spectroscopy,Japan,"[cobalt, energy supply]"
14,2043822537,10.1080/00958970500393158,"Potential propelling and rotating functions of propeller-type complexes. III. Detection of the propelling motion in <i>fac</i>-tris(<i>S</i>-2,3-diaminopropionato)cobalt(III) in an aqueous solution upon IR irradiation",2006,0.9982,G-Quadruplex DNA Structures and Functions,"Biochemistry, Genetics and Molecular Biology",Molecular Biology,Japan,"[cobalt, energy supply]"


In [ ]:
%%bigquery work_ids_a_max_fs_wip --project insyspo

SELECT a.*, f.abstract as abstract, c.display_name as topic, e.display_name AS field, d.display_name as subfield, i.country, h.work_id,
institution_id, h.author_id, j.display_name as author, k.funder_id, l.display_name as funder, i.display_name as institution
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.topics AS c ON b.topic_id = c.id
JOIN insyspo.publicdb_openalex_2024_10_rm.subfields AS d ON c.subfield = d.id
JOIN insyspo.publicdb_openalex_2024_10_rm.fields AS e ON c.field = e.id
JOIN insyspo.projectdb_cwts_summer_school_2023.works_abstracts f ON a.id = f.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works g ON a.id = g.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works_authorships AS h ON a.id = h.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.institutions AS i ON h.institution_id = i.id
JOIN insyspo.publicdb_openalex_2024_10_rm.authors AS j ON h.author_id = j.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works_grants AS k ON a.id = k.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.funders AS l ON k.funder_id = l.id




ERROR:
 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/insyspo/jobs?prettyPrint=false: Access Denied: Project insyspo: User does not have bigquery.jobs.create permission in project insyspo.

Location: None
Job ID: b86152fd-8bd2-498c-869d-4914d486dded



In [ ]:
df=work_ids_a_max_fs_wip
df[['id','doi','title','publication_year','max_score','topic','field','subfield','country','author','funder','institution']].head()

TypeError: 'QueryJob' object is not subscriptable

In [ ]:
author_works = (
    df.groupby('author')
      .apply(lambda x: list(zip(x['id'], x['title'])))
      .reset_index(name='works')
)

# add count as well
author_works['count'] = author_works['works'].apply(len)

# sort by count
author_works = author_works.sort_values('count', ascending=False)

author_works


#SUB-GRUPO: Conceitos + Conceitos + Contexto

##Coleta de dados da OpenAlex

###Extração de IDs



In [ ]:
from google.cloud import bigquery
from google.colab import auth
auth.authenticate_user()

client = bigquery.Client(project="insyspo")

In [ ]:
context = ["energy transition", "renewable energy","energy expansion", "energy system expansion", "electricity expansion", "energy planning",
           "energy demand", "energy supply", "energy infrastructure", "energy planning"]

problems = [
    "community displacement",
    "forced relocation", #"basic materials"
    "land use change",
    #"lithium triangle",
    "expropriation",
    "climate vulnerability"
    ,"biodiversity loss", "water pollution", "environmental conflict",
    "mining conflict", "resource conflict", "socio-environmental conflict",
    #"energy transition",
    #"renewab", "energy justice", "decarbon",
    #"sustainable supply", "low-carbon transition", "clean energy", "green energy",
    #"sustainable energy", "carbon neutral", "net zero", "carbon reduc", "carbon mitigat",
    #"energy secur", "climate resilien", "energy resilien", "resource sustainab",
    #"sustainable develop", "resource efficien",
    "just transition", "epistemic justice", "climate coloniality", "green sacrifice zone",
    "dispossession", "neoextractivism", "environmental degradation", "governance gap",
    "extractive frontier", "resource curse", #"mineral governance",
    "environmental injustice",
    "environmental violence", "technological dependence", "inclusion gap",
    "climate risk", #"financial risk",
    "transition risk",
    #"environmental liability",
    #"social license to operate",
    #"resource nationalism", "regulatory uncertainty",
    #"environmental regulation", "corporate accountability",
    "conflict mineral",
    "human rights", "environmental activism", "indigenous territory", "land rights",
    "cultural erosion", "territorial conflict", "conflict over resources", "community resistance",
    "environmental defender", "livelihood disruption", "public participation", "prior consultation",
    "traditional knowledge", "water scarcity", "soil contamination", "toxic waste",
    "ecosystem degradation", "health impacts", "social exclusion", #"regulatory capture",
    "climate injustice"
    # "displacement",
    # "territorial dispossession",
    # "land grabbing",
    # "water source",
    # "biodiversity",
    # "extractive conflict",
    # "extractive industry conflict",
    # "transition justice",
    # "social displacement",
    # "asymmetrical trade",
    # "commodity dependence",
    # "extractive dependency",
    # "carbon footprint shift",
    # "knowledge marginalization",
    # "governance failure",
    # "policy misalignment",
    # "community resistance",
    # "territorial justice",
    # "climate litigation",
    "green colonialism",
    #"mineral dependency",
    #"mineral sovereignty",
    #"green capitalism",
    #"green finance",
    "sacrifice zone",
    "climate justice",
    #"greenwashing",
    "technological dependency",
    #"sustainable finance",
    "green extractivism",
    #"global value chain",
    #"productive chain",
    #"resource gov"
]

problems1 = []

concepts = [
    "critical mineral",
    "critical metal",
    "critical material",
    "critical raw material",
    "strategic mineral",
    "strategic material",
    "key raw material",
    "rare metal",
    "green transition mineral",
    "green transition metal",
    "green transition material",
    "green transition raw material",
    #"strategic metal",
    "rare earth element",
    "battery metal",
    #"transition metal",
    #"mineral resource",
    "raw material dependenc",
    # "high-demand mineral",
    # "high-demand metal",
    #"material scarci",
    #"resource scarci",
    # "metal sustainab",
    #"mineral sustainab",
    #"resource critical",
    #"material critical",
    "aluminium", "antimony", "arsenic", "barite", "beryllium",
    "bismuth", "cerium", "cesium", "chromium", "cobalt", "dysprosium", "erbium", "europium", "fluorspar",
    "gadolinium", "gallium", "germanium", "graphite", "hafnium", "holmium", "indium", "iridium",
    "lanthanum", "lithium", "lutetium",
    #"magnesium",
    "manganese", "neodymium", "nickel", "niobium",
    "palladium", "platinum", "praseodymium", "rhodium", "rubidium", "ruthenium", "samarium",
    "scandium", "tantalum", "tellurium", "terbium", "thulium", "titanium", "tungsten",
    "uranium", "vanadium", "ytterbium", "yttrium", "zinc", "zirconium", "copper",
    #"clean tech",
    #"critical supply chain",
    #"supply chain vulnerability",
    "rare earth",
    #"battery supply chain",
    "energy storage material"
    #"strategic resource",
    #"low-carbon technology",
    # "raw material governance",           # ❌ nenhum match
    #"sustainable mining",
    #"circular mining",
    #"secondary sourcing",
    #"geopolitical risk",
    # "critical material recycling",       # ❌ nenhum match
    #"resource diplomacy",
    #"local content policy",
    #"net-zero strategy",
    #"carbon-neutral technology",
    #"renewable infrastructure"
]

def build_like_clauses(terms, field):
    if not terms:
        return ""
    return "\n                OR ".join([f"LOWER({field}) LIKE '%{term.lower()}%'" for term in terms])

# Build clauses
problem_display_clause = build_like_clauses(problems, "display_name")
concept_display_clause = build_like_clauses(concepts, "display_name")
context_display_clause = build_like_clauses(context, "display_name")

problem_abstract_clause = build_like_clauses(problems, "abstract")
concept_abstract_clause = build_like_clauses(concepts, "abstract")
context_abstract_clause = build_like_clauses(context, "abstract")

def make_where_clause(problem_clause, concept_clause, context_clause):
    clauses = []
    if problem_clause:
        clauses.append(f"({problem_clause})")
    if concept_clause:
        clauses.append(f"({concept_clause})")
    if context_clause:
        clauses.append(f"({context_clause})")
    return " AND ".join(clauses) if clauses else "TRUE"

works_where = make_where_clause(problem_display_clause, concept_display_clause, context_display_clause)
abstract_where = make_where_clause(problem_abstract_clause, concept_abstract_clause, context_abstract_clause)

query = f"""
DROP TABLE IF EXISTS `insyspo.userdb_fernando_spolidorio.cm_work_ids_3`;

CREATE TABLE `insyspo.userdb_fernando_spolidorio.cm_work_ids_3` AS (
    SELECT DISTINCT id
    FROM (
        SELECT id
        FROM `insyspo.publicdb_openalex_2024_10_rm.works`
        WHERE {works_where}
        UNION ALL
        SELECT id
        FROM `insyspo.projectdb_cwts_summer_school_2023.works_abstracts`
        WHERE {abstract_where}
    )
);
"""


In [ ]:
# Execute the query
job = client.query(query)
job.result()  # Wait for completion
print("Query completed and table created.")

Query completed and table created.


In [ ]:
%%bigquery work_ids_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
len(work_ids_3.drop_duplicates(subset=['id']))

261

###Filtros

####Artigos e anos

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS (

SELECT a.id, b.doi, b.title, b.type, b.publication_year
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works AS b ON a.id = b.id
WHERE b.type = 'article' --and b.publication_year > 2000 and b.publication_year < 2025
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Volume de trabalhos que são artigos.
len(work_ids_a_3.drop_duplicates(subset=['id']))

140

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS (

SELECT a.id, b.doi, b.title, b.type, b.publication_year
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works AS b ON a.id = b.id
WHERE b.type = 'article' and b.publication_year >= 2000 and b.publication_year < 2024
);

Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Volume de trabalhos que foram publicados entre 2000 e 2025
work_ids_a_3 = work_ids_a_3[
    (work_ids_a_3['publication_year'] > 2000) &
    (work_ids_a_3['publication_year'] < 2025)
]

len(work_ids_a_3)

107

####Maxscores

In [ ]:
%%bigquery --project insyspo

DROP TABLE IF EXISTS insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3;
CREATE TABLE insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS (

WITH MaxScores AS (
    SELECT a.*, MAX(b.score) AS max_score
    FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_3 AS a
    JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
    GROUP BY all
)

select * from maxscores

)


Query is running:   0%|          |

""


In [ ]:
%%bigquery work_ids_a_max_3 --project insyspo
select * from insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3

Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Artigos pós-dedupe
len(work_ids_a_max_3.drop_duplicates(subset=['id']))

106

###Fields & subfields

In [ ]:
%%bigquery work_ids_a_max_fs_3 --project insyspo

SELECT a.*, f.abstract as abstract, c.display_name as topic, e.display_name AS field, d.display_name as subfield, i.country, h.work_id,
institution_id, h.author_id, j.display_name as author, k.funder_id, l.display_name as funder, i.display_name as institution
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS a
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.topics AS c ON b.topic_id = c.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.subfields AS d ON c.subfield = d.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.fields AS e ON c.field = e.id
LEFT JOIN insyspo.projectdb_cwts_summer_school_2023.works_abstracts f ON a.id = f.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works g ON a.id = g.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_authorships AS h ON a.id = h.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.institutions AS i ON h.institution_id = i.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.authors AS j ON h.author_id = j.id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.works_grants AS k ON a.id = k.work_id
LEFT JOIN insyspo.publicdb_openalex_2024_10_rm.funders AS l ON k.funder_id = l.id



Query is running:   0%|          |

Downloading:   0%|          |

In [ ]:
#Verificando se o volume total de artigos está correto (precisa ser exatamente igual ao volume antes da adição de fields e subfields).
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

106

## Análise de palavras-chaves identificadas

###Criação de colunas

####Criação das colunas problems_kw e concept_kw

In [ ]:
import pandas as pd
import re

# --- 1. Exceções para match exato ---
exact_match_words = {"tin", "lead"}

# --- 2. Função de extração robusta (usa set para não duplicar) ---
def extract_keywords(text, keywords, exact_match_words):
    if pd.isna(text):
        return set()
    text = text.lower()
    found = set()
    for kw in keywords:
        kw_lc = kw.lower().strip()
        if kw_lc in exact_match_words:
            if re.search(rf'\b{re.escape(kw_lc)}\b', text):
                found.add(kw)
        else:
            if kw_lc in text:
                found.add(kw)
    return found

# --- 3. Normalizar listas de palavras-chave ---
problems  = [p.lower().strip() for p in problems]
concepts  = [c.lower().strip() for c in concepts]
contexts  = [c.lower().strip() for c in context]

# --- 4. Aplicar no DataFrame, unindo título e abstract SEM duplicar ---
work_ids_a_max_fs_3["problem_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], problems, exact_match_words)
        | extract_keywords(row["abstract"], problems, exact_match_words)
    ),
    axis=1
)

work_ids_a_max_fs_3["concept_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], concepts, exact_match_words)
        | extract_keywords(row["abstract"], concepts, exact_match_words)
    ),
    axis=1
)

work_ids_a_max_fs_3["context_kw"] = work_ids_a_max_fs_3.apply(
    lambda row: list(
        extract_keywords(row["title"], contexts, exact_match_words)
        | extract_keywords(row["abstract"], contexts, exact_match_words)
    ),
    axis=1
)

# --- 5. Identificar casos problemáticos ---
no_concept_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["concept_kw"].apply(lambda x: len(x) == 0)
]

no_problem_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["problem_kw"].apply(lambda x: len(x) == 0)
]

no_context_rows = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["context_kw"].apply(lambda x: len(x) == 0)
]

print("⚠️ Registros com ZERO conceitos detectados:", len(no_concept_rows))
print("⚠️ Registros com ZERO problemas detectados:", len(no_problem_rows))
print("⚠️ Registros com ZERO contextos detectados:", len(no_context_rows))


⚠️ Registros com ZERO conceitos detectados: 0
⚠️ Registros com ZERO problemas detectados: 0
⚠️ Registros com ZERO contextos detectados: 0


####Criação da coluna all_keywords

In [ ]:
work_ids_a_max_fs_3["all_keywords"] = work_ids_a_max_fs_3.apply(
    lambda row: sorted(set(row["problem_kw"] + row["concept_kw"] + row["context_kw"])),
    axis=1)

work_ids_a_max_fs_3[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of seawater into hydrogen and oxygen fuels",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[nickel, renewable energy, water scarcity]"
1,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of seawater into hydrogen and oxygen fuels",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[nickel, renewable energy, water scarcity]"
2,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of seawater into hydrogen and oxygen fuels",2019,1.0,Photocatalytic Materials for Solar Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[nickel, renewable energy, water scarcity]"
3,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of seawater into hydrogen and oxygen fuels",2019,1.0,Photocatalytic Materials for Solar Energy Conversion,Energy,"Renewable Energy, Sustainability and the Environment",United States,"[nickel, renewable energy, water scarcity]"
4,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of seawater into hydrogen and oxygen fuels",2019,1.0,Aqueous Zinc-Ion Battery Technology,Engineering,Electrical and Electronic Engineering,United States,"[nickel, renewable energy, water scarcity]"


In [ ]:
# salvar em Parquet
work_ids_a_max_fs_3.to_parquet(folder + 'subgroup_bronze.parquet', index=False)

#[SHORTCUT SUB-GROUP BRONZE]

In [ ]:
import pandas as pd
folder = '/content/drive/MyDrive/cientometria_resultados/Resultados preliminares/' #Fer's path
work_ids_a_max_fs_3 = pd.read_parquet(folder + 'subgroup_bronze.parquet')


In [ ]:
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

106

###20 combinações de kw mais comuns por área do conhecimento

In [ ]:
context = ["energy transition", "renewable energy","energy expansion", "energy system expansion", "electricity expansion", "energy planning",
           "energy demand", "energy supply", "energy infrastructure", "energy planning"]

problems = [
    "community displacement", "forced relocation", #"basic materials"
    "land use change",
    "lithium triangle", "expropriation",
    "climate vulnerability", "biodiversity loss", "water pollution", "environmental conflict",
    "mining conflict", "resource conflict", "socio-environmental conflict",
    #"energy transition",
    #"renewab", "energy justice", "decarbon",
    #"sustainable supply", "low-carbon transition", "clean energy", "green energy",
    #"sustainable energy", "carbon neutral", "net zero", "carbon reduc", "carbon mitigat",
    #"energy secur", "climate resilien", "energy resilien", "resource sustainab",
    #"sustainable develop", "resource efficien",
    "just transition", "epistemic justice", "climate coloniality", "green sacrifice zone",
    "dispossession", "neoextractivism", "environmental degradation", "governance gap",
    "extractive frontier", "resource curse", "mineral governance", "environmental injustice",
    "environmental violence", "technological dependence", "inclusion gap",
    "climate risk", "financial risk", "transition risk", "environmental liability",
    "social license to operate", "resource nationalism", "regulatory uncertainty",
    "environmental regulation", "corporate accountability", "conflict mineral",
    "human rights", "environmental activism", "indigenous territory", "land rights",
    "cultural erosion", "territorial conflict", "conflict over resources", "community resistance",
    "environmental defender", "livelihood disruption", "public participation", "prior consultation",
    "traditional knowledge", "water scarcity", "soil contamination", "toxic waste",
    "ecosystem degradation", "health impacts", "social exclusion", "regulatory capture",
    "climate injustice"
    # "displacement",
    # "territorial dispossession",
    # "land grabbing",
    # "water source",
    # "biodiversity",
    # "extractive conflict",
    # "extractive industry conflict",
    # "transition justice",
    # "social displacement",
    # "asymmetrical trade",
    # "commodity dependence",
    # "extractive dependency",
    # "carbon footprint shift",
    # "knowledge marginalization",
    # "governance failure",
    # "policy misalignment",
    # "community resistance",
    # "territorial justice",
    # "climate litigation",
    "green colonialism",
    "mineral dependency",
    #"mineral sovereignty",
    "green capitalism",
    "green finance",
    "sacrifice zone",
    "climate justice",
    "greenwashing",
    "technological dependency",
    "sustainable finance",
    "green extractivism",
    "global value chain",
    "productive chain",
    "resource gov"
]

problems1 = []

concepts = [
    "critical mineral",
    "critical metal",
    "critical material",
    "critical raw material",
    "strategic mineral",
    "strategic material",
    "key raw material",
    "rare metal",
    "green transition mineral",
    "green transition metal",
    "green transition material",
    "green transition raw material",
    #"strategic metal",
    "rare earth element",
    "battery metal",
    #"transition metal",
    #"mineral resource",
    "raw material dependenc",
    # "high-demand mineral",
    # "high-demand metal",
    #"material scarci",
    #"resource scarci",
    # "metal sustainab",
    #"mineral sustainab",
    #"resource critical",
    #"material critical",
    "aluminium", "antimony", "arsenic", "barite", "beryllium",
    "bismuth", "cerium", "cesium", "chromium", "cobalt", "dysprosium", "erbium", "europium", "fluorspar",
    "gadolinium", "gallium", "germanium", "graphite", "hafnium", "holmium", "indium", "iridium",
    "lanthanum", "lithium", "lutetium",
    #"magnesium",
    "manganese", "neodymium", "nickel", "niobium",
    "palladium", "platinum", "praseodymium", "rhodium", "rubidium", "ruthenium", "samarium",
    "scandium", "tantalum", "tellurium", "terbium", "thulium", "titanium", "tungsten",
    "uranium", "vanadium", "ytterbium", "yttrium", "zinc", "zirconium", "copper",
    #"clean tech",
    #"critical supply chain",
    #"supply chain vulnerability",
    "rare earth",
    #"battery supply chain",
    "energy storage material"
    #"strategic resource",
    #"low-carbon technology",
    # "raw material governance",           # ❌ nenhum match
    #"sustainable mining",
    #"circular mining",
    #"secondary sourcing",
    #"geopolitical risk",
    # "critical material recycling",       # ❌ nenhum match
    #"resource diplomacy",
    #"local content policy",
    #"net-zero strategy",
    #"carbon-neutral technology",
    #"renewable infrastructure"
]

df=work_ids_a_max_fs_3
df["all_keywords"] = df["all_keywords"].apply(lambda x: tuple(x))  # Convert lists to tuples

df_grouped = (
    df.groupby(["field", "all_keywords"])
    .size()
    .reset_index(name="count")
)

# Get top 10 counts per field
df_top10_3 = df_grouped.groupby("field", group_keys=False).apply(lambda x: x.nlargest(20, "count"))

df_top10_3


/tmp/ipython-input-3340289721.py:133: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



,field,all_keywords,count
0,Agricultural and Biological Sciences,"(copper, energy demand, soil contamination)",12
1,Arts and Humanities,"(cobalt, copper, energy transition, extractive...",1
2,Arts and Humanities,"(environmental activism, environmental degrada...",1
7,"Business, Management and Accounting","(energy demand, rare earth, toxic waste)",3
3,"Business, Management and Accounting","(arsenic, energy transition, environmental act...",1
...,...,...,...
179,Social Sciences,"(climate justice, environmental injustice, lit...",2
180,Social Sciences,"(climate risk, copper, energy infrastructure, ...",2
183,Social Sciences,"(cobalt, energy supply, energy transition, hum...",2
198,Social Sciences,"(environmental activism, environmental degrada...",2


###Volume e representatividade de palavras-chaves de Problema



In [ ]:
from collections import Counter
import pandas as pd

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# Flatten all lists of keywords into one big list
all_keywords_flat = work_ids_a_max_fs_3['problem_kw'].dropna().explode()

# Count keyword frequencies
keyword_counts = Counter(all_keywords_flat)

# Convert to DataFrame
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['keyword', 'count'])

# Compute total count of all keywords
total_keywords = keyword_df['count'].sum()

# Add percentage column
keyword_df['percent'] = (keyword_df['count'] / total_keywords * 100).round(1)

# Sort by most common
keyword_df = keyword_df.sort_values(by='count', ascending=False).reset_index(drop=True)

keyword_df

,keyword,count,percent
0,environmental degradation,22,15.9
1,water scarcity,16,11.6
2,human rights,15,10.9
3,water pollution,11,8.0
4,just transition,10,7.2
5,health impacts,6,4.3
6,public participation,5,3.6
7,sacrifice zone,5,3.6
8,environmental activism,4,2.9
9,conflict mineral,4,2.9


###Evolução Temporal das Principais Palavras-chave de Problema

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import textwrap

# ===== Parâmetros =====
keyword_col = 'problem_kw'
TOP_K = 10          # reduzir para melhorar a leitura
YEAR_START = 2000  # ajuste se quiser

# 0) Base deduplicada
work_ids_a_max_fs_3 = work_ids_a_max_fs_3.drop_duplicates(subset=['id'])

# 1) Explodir keywords e contar por ano + keyword
exploded_df = (
    work_ids_a_max_fs_3[['publication_year', keyword_col]]
    .explode(keyword_col)
    .dropna()
)

keyword_year_counts = (
    exploded_df
    .groupby(['publication_year', keyword_col])
    .size()
    .reset_index(name='count')
)

# 2) Selecionar TOP_K no período inteiro
top_keywords = (
    keyword_year_counts
    .groupby(keyword_col)['count']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_K)
    .index.tolist()
)

df_top = keyword_year_counts[keyword_year_counts[keyword_col].isin(top_keywords)].copy()

# 3) Pivot anos x palavras e garantir anos contínuos
pivot = df_top.pivot_table(index='publication_year', columns=keyword_col,
                           values='count', aggfunc='sum', fill_value=0)

if len(pivot):
    full_years = pd.Index(range(max(YEAR_START, int(pivot.index.min())),
                                int(pivot.index.max()) + 1), name='publication_year')
else:
    full_years = pd.Index([], name='publication_year')

pivot = pivot.reindex(full_years, fill_value=0)

# Ordenar colunas por total (maiores primeiro)
pivot = pivot[pivot.sum().sort_values(ascending=False).index]

# 4) Paleta em tons de cinza e padrões (texturas) para PB
gray_palette = [
    'rgba(0,0,0,1)', 'rgba(60,60,60,1)', 'rgba(100,100,100,1)',
    'rgba(140,140,140,1)', 'rgba(170,170,170,1)', 'rgba(200,200,200,1)',
    'rgba(120,120,120,1)', 'rgba(30,30,30,1)'
]
patterns = ['.', '/', '\\', 'x', '+', '-', '|', '']  # pontos, listras, cruzes…

# 5) Abreviar rótulos longos na legenda
def short(s, n=24):
    return '\n'.join(textwrap.wrap(str(s), width=n))

fig = go.Figure()

for i, col in enumerate(pivot.columns):
    fig.add_trace(
        go.Bar(
            x=pivot.index.astype(str),
            y=pivot[col],
            name=short(col, 24),
            marker=dict(
                color=gray_palette[i % len(gray_palette)],
                line=dict(color='black', width=0.5),
                pattern=dict(shape=patterns[i % len(patterns)], solidity=0.3)
            ),
            hovertemplate="Ano: %{x}<br>Frequência: %{y}<extra>" + str(col) + "</extra>"
        )
    )

fig.update_layout(
    barmode='stack',
    title="Evolução temporal das 10 principais palavras-chave de problema",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=560,
    margin=dict(l=70, r=40, t=70, b=110),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        tickangle=0,
        showgrid=False, gridcolor='rgba(0,0,0,0.12)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Frequência",
        rangemode='tozero',
        tickmode='linear', dtick=5,  # somente inteiros
        showgrid=True, gridcolor='rgba(0,0,0,0.12)',
        linecolor='black', mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.22,
        xanchor='center', x=0.5,
        traceorder='normal',
        bgcolor='rgba(255,255,255,0.8)'
    )
)

fig.show()


In [ ]:
pivot

problem_kw,environmental degradation,water scarcity,human rights,water pollution,just transition,health impacts,sacrifice zone,public participation,environmental activism,resource curse
publication_year,,,,,,,,,,
2008,0,0,1,0,0,0,0,0,0,0
2009,0,0,0,0,0,0,0,0,0,0
2010,0,0,0,2,0,0,0,0,0,0
2011,0,0,0,1,0,0,0,1,0,0
2012,0,0,0,0,0,0,0,0,1,0
2013,1,0,0,0,0,0,0,0,0,0
2014,1,0,0,1,0,0,0,0,0,0
2015,1,2,0,0,0,0,0,0,0,0
2016,0,1,0,1,0,2,0,1,0,0


###Volume e representatividade de palavras-chaves de Conceitos

In [ ]:
from collections import Counter
import pandas as pd

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# Flatten all lists of keywords into one big list
all_keywords_flat = work_ids_a_max_fs_3['concept_kw'].dropna().explode()

# Count keyword frequencies
keyword_counts = Counter(all_keywords_flat)

# Convert to DataFrame
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['keyword', 'count'])

# Compute total count of all keywords
total_keywords = keyword_df['count'].sum()

# Add percentage column
keyword_df['percent'] = (keyword_df['count'] / total_keywords * 100).round(1)

# Sort by most common
keyword_df = keyword_df.sort_values(by='count', ascending=False).reset_index(drop=True)

# Show full result without

keyword_df

,keyword,count,percent
0,lithium,36,17.4
1,nickel,21,10.1
2,copper,20,9.7
3,cobalt,15,7.2
4,rare earth,15,7.2
5,uranium,10,4.8
6,graphite,8,3.9
7,titanium,7,3.4
8,critical mineral,7,3.4
9,critical raw material,6,2.9


###Evolução Temporal das Principais Palavras-chave de Conceito

In [ ]:
import pandas as pd
import plotly.graph_objects as go

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# ===== Parâmetros =====
keyword_col = 'concept_kw'
TOP_K = 20

# 1) Explodir keywords e contar por ano + keyword
exploded_df = (
    work_ids_a_max_fs_3[['publication_year', keyword_col]]
    .explode(keyword_col)
    .dropna()
)

keyword_year_counts = (
    exploded_df
    .groupby(['publication_year', keyword_col])
    .size()
    .reset_index(name='count')
)

# 2) Selecionar as TOP_K palavras mais frequentes no período
top_keywords = (
    keyword_year_counts
    .groupby(keyword_col)['count']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_K)
    .index.tolist()
)

df_top = keyword_year_counts[keyword_year_counts[keyword_col].isin(top_keywords)].copy()

# 3) Pivot (anos x palavras) e ordenação por total
pivot = (
    df_top.pivot_table(index='publication_year', columns=keyword_col,
                       values='count', aggfunc='sum', fill_value=0)
          .sort_index()
)
pivot = pivot[pivot.sum().sort_values(ascending=False).index]

# 4) Símbolos e traços para 20 séries (10 marcadores × 2 dashes = 20 combinações)
marker_symbols = [
    'circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
    'triangle-left', 'triangle-right', 'cross', 'x', 'star'
]
dash_styles = ['solid', 'dash']  # duas variações de traço
styles = [(m, d) for d in dash_styles for m in marker_symbols]  # 20 combinações

# 5) Figura Plotly (ABNT/DPCT-IG)
fig = go.Figure()

for (col, (marker_sym, dash)) in zip(pivot.columns, styles):
    fig.add_trace(
        go.Scatter(
            x=pivot.index.astype(str),  # anos como categorias ordenadas
            y=pivot[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.5, dash=dash),
            marker=dict(
                symbol=marker_sym,
                size=6,
                color='white',                   # preenchimento branco
                line=dict(color='black', width=1) # contorno preto
            ),
            hovertemplate="Ano: %{x}<br>Frequência: %{y}<extra>"+str(col)+"</extra>"
        )
    )

fig.update_layout(
    title="Evolução temporal das 20 principais palavras-chave de conceito",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Frequência da palavra-chave",
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.30,    # legenda abaixo do gráfico
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


###Volume e representatividade de palavras-chaves de Contexto

In [ ]:
from collections import Counter
import pandas as pd

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# Flatten all lists of keywords into one big list
all_keywords_flat = work_ids_a_max_fs_3['context_kw'].dropna().explode()

# Count keyword frequencies
keyword_counts = Counter(all_keywords_flat)

# Convert to DataFrame
keyword_df = pd.DataFrame(keyword_counts.items(), columns=['keyword', 'count'])

# Compute total count of all keywords
total_keywords = keyword_df['count'].sum()

# Add percentage column
keyword_df['percent'] = (keyword_df['count'] / total_keywords * 100).round(1)

# Sort by most common
keyword_df = keyword_df.sort_values(by='count', ascending=False).reset_index(drop=True)

# Show full result without truncation
with pd.option_context('display.max_rows', None):
    print(keyword_df)

keyword_df

                 keyword  count  percent
0       renewable energy     71     53.4
1      energy transition     28     21.1
2          energy demand     18     13.5
3          energy supply     13      9.8
4  energy infrastructure      2      1.5
5  electricity expansion      1      0.8


,keyword,count,percent
0,renewable energy,71,53.4
1,energy transition,28,21.1
2,energy demand,18,13.5
3,energy supply,13,9.8
4,energy infrastructure,2,1.5
5,electricity expansion,1,0.8


###Evolução Temporal das Principais Palavras-chave de Contexto

In [ ]:
import pandas as pd
import plotly.graph_objects as go

work_ids_a_max_fs_3=work_ids_a_max_fs_3.drop_duplicates(subset=['id']) #DESDUPLICAÇÃO DE ARTIGOS (MANTÉM SOMENTE LISTA DE IDs ÚNICOS)

# ===== Parâmetros =====
keyword_col = 'context_kw'
TOP_K = 10

# 1) Explodir keywords e contar por ano + keyword
exploded_df = (
    work_ids_a_max_fs_3[['publication_year', keyword_col]]
    .explode(keyword_col)
    .dropna()
)

keyword_year_counts = (
    exploded_df
    .groupby(['publication_year', keyword_col])
    .size()
    .reset_index(name='count')
)

# 2) Selecionar as TOP_K palavras mais frequentes no período
top_keywords = (
    keyword_year_counts
    .groupby(keyword_col)['count']
    .sum()
    .sort_values(ascending=False)
    .head(TOP_K)
    .index.tolist()
)

df_top = keyword_year_counts[keyword_year_counts[keyword_col].isin(top_keywords)].copy()

# 3) Pivot (anos x palavras) e ordenação por total
pivot = (
    df_top.pivot_table(index='publication_year', columns=keyword_col,
                       values='count', aggfunc='sum', fill_value=0)
          .sort_index()
)
pivot = pivot[pivot.sum().sort_values(ascending=False).index]  # maiores primeiro

# 4) Marcadores distintos (10 símbolos)
marker_symbols = [
    'circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
    'triangle-left', 'triangle-right', 'cross', 'x', 'star'
]

# 5) Figura Plotly (ABNT/DPCT-IG)
fig = go.Figure()

for (col, sym) in zip(pivot.columns, marker_symbols):
    fig.add_trace(
        go.Scatter(
            x=pivot.index.astype(str),            # anos como categorias ordenadas
            y=pivot[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.8, dash='solid'),  # linhas pretas sólidas
            marker=dict(
                symbol=sym,
                size=7,
                color='white',                     # preenchimento branco
                line=dict(color='black', width=1)  # contorno preto (bom para PB)
            ),
            hovertemplate="Ano: %{x}<br>Frequência: %{y}<extra>"+str(col)+"</extra>"
        )
    )

fig.update_layout(
    title="Evolução temporal das 10 principais palavras-chave de contexto",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=600,
    margin=dict(l=80, r=80, t=80, b=140),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Frequência da palavra-chave",
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    showlegend=True,
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,   # legenda abaixo
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


###Volume de kw por área

In [ ]:
import pandas as pd

# Step 1: Calculate total count per field
field_order = (
    df_top10_3.groupby('field')['count']
    .sum()
    .sort_values(ascending=False)
    .index
)

# Step 2: Sort df_top10_3 by field (with custom order) and then by count descending
df_top10_3_sorted = (
    df_top10_3
    .assign(field=pd.Categorical(df_top10_3['field'], categories=field_order, ordered=True))
    .sort_values(by=['field', 'count'], ascending=[True, False])
    .reset_index(drop=True)
)

df_top10_3_sorted

,field,all_keywords,count
0,Engineering,"(nickel, renewable energy, water scarcity)",168
1,Engineering,"(energy demand, graphite, water scarcity)",48
2,Engineering,"(lithium, renewable energy, water scarcity)",48
3,Engineering,"(health impacts, nickel, renewable energy, tit...",36
4,Engineering,"(energy demand, environmental degradation, ren...",32
...,...,...,...
128,Health Professions,"(copper, energy demand, health impacts, renewa...",1
129,Arts and Humanities,"(cobalt, copper, energy transition, extractive...",1
130,Arts and Humanities,"(environmental activism, environmental degrada...",1
131,Medicine,"(aluminium, energy demand, energy supply, envi...",1


###Consulta de kw para consultar artigos

####Consulta de kw não encontradas

In [ ]:
all_found_problems = set([kw.lower() for sublist in work_ids_a_max_fs_3["problem_kw"] for kw in sublist])
all_found_concepts = set([kw.lower() for sublist in work_ids_a_max_fs_3["concept_kw"] for kw in sublist])
all_found_context = set([kw.lower() for sublist in work_ids_a_max_fs_3["context_kw"] for kw in sublist])

# Quais não foram encontradas?
problems_not_found = sorted(set(problems) - all_found_problems)
concepts_not_found = sorted(set(concepts) - all_found_concepts)
context_not_found = sorted(set(context) - all_found_context)

# Mostrar resultados
print("❌ Palavras-chave de 'problems' que não tiveram nenhum match:")
print(problems_not_found)

print("\n❌ Palavras-chave de 'concepts' que não tiveram nenhum match:")
print(concepts_not_found)

print("\n❌ Palavras-chave de 'context' que não tiveram nenhum match:")
print(context_not_found)

❌ Palavras-chave de 'problems' que não tiveram nenhum match:
['climate injusticegreen colonialism', 'climate vulnerability', 'community resistance', 'conflict over resources', 'corporate accountability', 'cultural erosion', 'ecosystem degradation', 'environmental defender', 'environmental liability', 'environmental regulation', 'environmental violence', 'epistemic justice', 'financial risk', 'forced relocation', 'global value chain', 'governance gap', 'green capitalism', 'green finance', 'greenwashing', 'inclusion gap', 'land rights', 'lithium triangle', 'livelihood disruption', 'mineral dependency', 'mineral governance', 'mining conflict', 'neoextractivism', 'prior consultation', 'productive chain', 'regulatory capture', 'regulatory uncertainty', 'resource conflict', 'resource gov', 'resource nationalism', 'social license to operate', 'socio-environmental conflict', 'sustainable finance', 'technological dependence', 'technological dependency', 'territorial conflict', 'traditional know

In [ ]:
from collections import Counter
import pandas as pd
pd.set_option('display.max_colwidth', None)

# Define the keyword to check
keyword_to_check = "epistemic justice"

# Ensure all_keywords is a list
work_ids_a_max_fs_3["all_keywords"] = work_ids_a_max_fs_3["all_keywords"].apply(
    lambda x: x if isinstance(x, list) else []
)

# Filter rows that include the keyword
filtered = work_ids_a_max_fs_3[
    work_ids_a_max_fs_3["all_keywords"].apply(lambda x: keyword_to_check in x)
]

# Convert list of keywords to tuple (for grouping), preserving order (or use sorted(x) if order doesn’t matter)
filtered["keyword_combination"] = filtered["all_keywords"].apply(lambda x: tuple(x))

# Group by field and keyword combination
grouped = (
    filtered.groupby(["field", "keyword_combination", "title", "abstract"])
    .size()
    .reset_index(name="count")
    .sort_values(by="count", ascending=False)
)

# Show result
grouped


,field,keyword_combination,title,abstract,count


# Group BERTopic filtering

###Setup

In [ ]:
!pip install bertopic

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import random

from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer

work_ids_a_max_fs_3=pd.read_parquet(folder+'subgroup_bronze.parquet')

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired

# Step 1 - Embeddings (good baseline; consider multilingual if PT/EN mixed)
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Step 2 - UMAP: fewer neighbors -> more local structure (more/smaller clusters)
umap_model = UMAP(
    n_neighbors=15,          # was 15
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

# Step 3 - HDBSCAN: split more aggressively
hdbscan_model = HDBSCAN(
    min_cluster_size=5,              # was 15
    min_samples=1,                   # lower = easier to split
    metric="euclidean",              # you can try "cosine" too
    cluster_selection_method="leaf", # finer splits than "eom"
    prediction_data=True
)

# Step 4 - Vectorizer: keep informative terms; allow bigrams
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),
    min_df=3,                        # ignore ultra-rare terms; adjust to corpus size
    stop_words="english"             # add domain stopwords if needed
)

# Step 5 - c-TF-IDF
ctfidf_model = ClassTfidfTransformer()

# Step 6 - Representation model
representation_model = KeyBERTInspired()

# Assemble BERTopic with granularity-friendly options
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    language="english",
    nr_topics=None,                  # do NOT merge topics (keeps granularity)
    low_memory=True,
    verbose=True
)

###Topics

In [ ]:
import pandas as pd

# 1) Prep: one row per (title, abstract) and keep metadata
prep = work_ids_a_max_fs_3.drop_duplicates(subset=['title', 'abstract'])

meta = (
    prep[['id', 'title', 'abstract']]
    .copy()
)

# Drop rows with nulls or empty/whitespace-only title/abstract
meta[['title', 'abstract']] = meta[['title', 'abstract']].astype('string')
meta = (
    meta
    .replace({'title': r'^\s*$', 'abstract': r'^\s*$'}, pd.NA, regex=True)
    .dropna(subset=['id', 'title', 'abstract'])
    .reset_index(drop=True)
)


In [ ]:
len(meta)

117

In [ ]:
meta = meta.iloc[random.sample(range(len(meta)), min(4000, len(meta)))]

In [ ]:
# 2) Build docs (list[str]) for BERTopic from title + abstract
meta['doc'] = (
    meta[['title', 'abstract']]
    .agg('. '.join, axis=1)               # join with period + space
    .str.replace(r'\s+', ' ', regex=True) # collapse whitespace
    .str.strip()
)

# Keep only non-empty docs and preserve alignment
mask = meta['doc'].ne('')
meta = meta.loc[mask].reset_index(drop=True)
docs = meta['doc'].tolist()


In [ ]:
# 3) Fit / transform
topics, probs = topic_model.fit_transform(docs)

2025-10-19 19:19:06,189 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2025-10-19 19:19:44,570 - BERTopic - Embedding - Completed ✓
2025-10-19 19:19:44,573 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-10-19 19:19:45,293 - BERTopic - Dimensionality - Completed ✓
2025-10-19 19:19:45,297 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-10-19 19:19:45,349 - BERTopic - Cluster - Completed ✓
2025-10-19 19:19:45,362 - BERTopic - Representation - Fine-tuning topics using representation models.
2025-10-19 19:19:48,692 - BERTopic - Representation - Completed ✓


In [ ]:
# Mantém o gráfico base
fig_barchart = topic_model.visualize_barchart(
    width=280,
    height=330,
    top_n_topics=20,
    n_words=10
)

# --- Ajustes de estilo ABNT/DPCT ---
fig_barchart.update_traces(
    marker_color="gray",         # cor única
    marker_line_color="black",   # contorno preto fino
    marker_line_width=0.5,       # espessura discreta
    opacity=0.85                 # leve transparência
)

fig_barchart.update_layout(
    title_text="Tópicos Encontrados",
    font=dict(family="Times New Roman", size=12),
    title=dict(font=dict(size=14)),
    plot_bgcolor="white",        # fundo branco
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=20, t=60, b=60)
)

# renomeia os títulos de cada subplot
for i, ann in enumerate(fig_barchart.layout.annotations):
    ann.text = f"Tópico {i+1}"
    ann.font = dict(family="Times New Roman", size=11)

fig_barchart.show()


In [ ]:
document_info = topic_model.get_document_info(docs).copy()  # expects the same list[str]
document_info['id'] = meta['id'].to_numpy()                 # align by position

In [ ]:
neg_topic=document_info[document_info['Topic'] == -1]
#neg_topic[['Topic', 'Name','Top_n_words','Probability','Representative_document','id']]
neg_topic['Top_n_words'].unique()

array(['renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas'],
      dtype=object)

In [ ]:
ids=neg_topic['id'].unique()
ids=pd.DataFrame(ids, columns = ['id'])
ids

,id
0,4389961977
1,317249264
2,2248528710
3,4388540217
4,4214673554
5,3195645655
6,1534487135
7,2789421862
8,2909377867
9,4362582143


In [ ]:
filtered = work_ids_a_max_fs_3[work_ids_a_max_fs_3['id'].isin(ids['id'])]
filtered[['id','title','abstract']].drop_duplicates(subset=['id'])

id  \
8    1534487135   
9    4362582143   
15   4214673554   
38   2969553505   
55   4307228855   
59   4311208886   
65   4210687709   
71    317249264   
77   4388540217   
80   4390034052   
84   3195645655   
100  2789421862   
104  4389961977   
108  2248528710   
109  2909377867   

                                                                                                                                                        title  \
8                                   Remote sensing and geochemistry techniques for the assessment of coal mining pollution, Emalahleni (Witbank), Mpumalanga.   
9                                                 Global Environmental Health Impacts of Rare Earth Metals: Insights for Research and Policy Making in Africa   
15                                                                              Resource Depletion Risk for Medical Equipment: Embracing the Circular Economy   
38                                                                          The geopolitics of metals and metalloids used for the renewable energy transition   
55                         Hotspots of Mining-Related Biodiversity Loss in Global Supply Chains and the Potential for Reduction through Renewable Electricity   
59                                                                    Culture Shock: the legacy of the 1960s power generation schemes in Aotearoa New Zealand   
65             Critical sustainability issues in the production of wind and solar electricity generation as well as storage facilities and possible solutions   
71                                                                                          Overview of Recent Development in Energy Resources Law in Nigeria   
77                                                                 Megaprojects on ice: lessons from the Kárahnjúkar hydropower project for a Just Transition   
80                                                                    Nickel mining in Obi Island: problem or solution for mitigating climate change effects?   
84   Valorisation of mine waste - Part II: Resource evaluation for consolidated and mineralised mine waste using the Central African Copperbelt as an example   
100                                The social footprint of permanent magnet production based on rare earth elements – a social life cycle assessment scenario   
104                                                                                       Cobalt in Battery Production: Implications for the Mining Community   
108                                                                             Engineering Solutions to Storm Water Problems Through Community Participation   
109                                                                                                                                  The Legacy of Extraction   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
# 4) Get per-document info and reattach IDs
document_info = topic_model.get_document_info(docs).copy()  # expects the same list[str]
document_info['id'] = meta['id'].to_numpy()                 # align by position

In [ ]:
# 5) Retrieve all IDs with Topic > 0  (exclude outliers -1 and topic 0)
ids_topic_gt0 = document_info.loc[document_info['Topic'] >= 0, 'id'].tolist()

###Saves topics

In [ ]:
topic_model.save(folder+'subgroup_topics')

2025-10-19 19:19:49,530 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [ ]:
#Salva os tópicos únicos e suas informações
unique_topics=document_info['Topic'].unique()
unique_topics=document_info[document_info['Topic'].isin(unique_topics)]
unique_topics=unique_topics[['id','Topic','Name','Top_n_words']].drop_duplicates(subset=['id','Topic','Top_n_words'])
df=unique_topics.sort_values(by='Topic')
df.to_parquet(folder + 'subgroup_bronze_topics.parquet', index=False)

#Group topics shortcut

###Visualização dos tópicos

In [ ]:
from bertopic import BERTopic
import os

topic_model = BERTopic.load(os.path.join(folder, 'subgroup_topics'))

In [ ]:
# Mantém o gráfico base
fig_barchart = topic_model.visualize_barchart(
    width=280,
    height=330,
    top_n_topics=20,
    n_words=10
)

# --- Ajustes de estilo ABNT/DPCT ---
fig_barchart.update_traces(
    marker_color="gray",         # cor única
    marker_line_color="black",   # contorno preto fino
    marker_line_width=0.5,       # espessura discreta
    opacity=0.85                 # leve transparência
)

fig_barchart.update_layout(
    title_text="Tópicos Encontrados",
    font=dict(family="Times New Roman", size=12),
    title=dict(font=dict(size=14)),
    plot_bgcolor="white",        # fundo branco
    paper_bgcolor="white",
    showlegend=False,
    margin=dict(l=60, r=20, t=60, b=60)
)

# renomeia os títulos de cada subplot
for i, ann in enumerate(fig_barchart.layout.annotations):
    ann.text = f"Tópico {i+1}"
    ann.font = dict(family="Times New Roman", size=11)

fig_barchart.show()


##Distribuição de tópicos

In [ ]:
import pandas as pd
df=pd.read_parquet(folder + 'subgroup_bronze_topics.parquet')
df

,id,Topic,Name,Top_n_words
0,4389961977,-1,-1_renewable energy_renewable_electricity_coal,renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas
1,4388540217,-1,-1_renewable energy_renewable_electricity_coal,renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas
2,2248528710,-1,-1_renewable energy_renewable_electricity_coal,renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas
3,317249264,-1,-1_renewable energy_renewable_electricity_coal,renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas
4,3195645655,-1,-1_renewable energy_renewable_electricity_coal,renewable energy - renewable - electricity - coal - mining - waste - energy - environmental - cobalt - natural gas
...,...,...,...,...
112,2112408723,8,8_water scarcity_environmental impacts_environmental impact_emissions,water scarcity - environmental impacts - environmental impact - emissions - energy - global warming - environmental - recycling - electricity - scarcity
113,3032922613,8,8_water scarcity_environmental impacts_environmental impact_emissions,water scarcity - environmental impacts - environmental impact - emissions - energy - global warming - environmental - recycling - electricity - scarcity
114,2300464399,8,8_water scarcity_environmental impacts_environmental impact_emissions,water scarcity - environmental impacts - environmental impact - emissions - energy - global warming - environmental - recycling - electricity - scarcity
115,4285398234,8,8_water scarcity_environmental impacts_environmental impact_emissions,water scarcity - environmental impacts - environmental impact - emissions - energy - global warming - environmental - recycling - electricity - scarcity


In [ ]:
import pandas as pd

# Group by topic name and top words, counting how many articles per group
topic_summary = (
    df[df['Topic'] > -1]  # filter out the outlier topic (-1)
    .groupby(['Topic', 'Name', 'Top_n_words'])
    .agg(article_count=('id', 'count'))
    .reset_index()
    .sort_values('article_count', ascending=False)
)

topic_summary


,Topic,Name,Top_n_words,article_count
0,0,0_renewable energy_renewable_fuels_recycling,renewable energy - renewable - fuels - recycling - energy generation - environmental - pollution - carbon - environmental degradation - fuel,21
1,1,1_lithium mining_renewable energy_renewable_lithium,lithium mining - renewable energy - renewable - lithium - sustainability - climate change - mining - energy - minerals - environmental,19
2,2,2_renewable energy_renewable_emissions_environmental,renewable energy - renewable - emissions - environmental - hydrogen - fuel - sustainable - carbon - sustainability - natural gas,17
3,3,3_energy water_renewable_energy resources_water resources,energy water - renewable - energy resources - water resources - environmental - coal - environmental impacts - groundwater - land use - electrolysis,10
4,4,4_nuclear power_environmental protection_environment_coal,nuclear power - environmental protection - environment - coal - environmental - fuels - infrastructure - energy - environmental regulations - climate change,10
5,5,5_renewable_renewable energy_policy_policies,renewable - renewable energy - policy - policies - risks - industrial - critical minerals - economic - minerals - supply chains,8
6,6,6_renewable energy_energy storage_sustainable_water scarcity,renewable energy - energy storage - sustainable - water scarcity - solar energy - energy - hydrogen - electricity - scarcity - earth,6
7,7,7_environmental impacts_environmental_emissions_raw materials,environmental impacts - environmental - emissions - raw materials - environment - nuclear power - industrial - energy - energy consumption - uranium,6
8,8,8_water scarcity_environmental impacts_environmental impact_emissions,water scarcity - environmental impacts - environmental impact - emissions - energy - global warming - environmental - recycling - electricity - scarcity,5


In [ ]:
import plotly.express as px

# Sort by article_count (descending) and keep top 30 for readability
topic_summary_top30 = topic_summary.sort_values('article_count', ascending=False).head(30)

# Create bar chart (monochrome / ABNT style)
fig = px.bar(
    topic_summary_top30,
    x='Name',
    y='article_count',
    text='article_count',
    title='Distribuição de artigos por tópico (Top 30)',
)

# Style: grayscale, Times New Roman, axis lines
fig.update_traces(
    textposition='outside',
    marker_color='rgba(80,80,80,1)',
    marker_line_color='black',
    marker_line_width=0.8
)

fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=600,
    margin=dict(l=60, r=40, t=80, b=140),
    xaxis=dict(
        title='Tópico',
        tickangle=45,
        linecolor='black',
        mirror=True,
        showgrid=False,
    ),
    yaxis=dict(
        title='Número de artigos',
        tickmode='linear',
        dtick=5,  # mínimo espaçamento de 5 entre ticks
        linecolor='black',
        mirror=True,
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
    ),
    showlegend=False,
)

fig.show()


##IDs para análise

In [ ]:
ids_df = pd.DataFrame(ids_topic_gt0, columns=['id'])
len(ids_df)

102

In [ ]:
ids_df.to_parquet(folder + 'subgroup_gold.parquet', index=False)

#[SHORTCUT SUB-GROUP GOLD]

In [ ]:
import pandas as pd

folder = '/content/drive/MyDrive/cientometria_resultados/Resultados preliminares/' #Fer's path
work_ids_a_max_fs_3 = pd.read_parquet(folder + 'subgroup_bronze.parquet')
ids_df=pd.read_parquet(folder + 'subgroup_gold.parquet')

In [ ]:
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

106

In [ ]:
len(ids_df.drop_duplicates(subset=['id']))

102

##Análise de publicações

###Publicações por área do conhecimento

In [ ]:
ids_set = set(ids_df['id'])
work_ids_a_max_fs_3 = work_ids_a_max_fs_3[work_ids_a_max_fs_3['id'].isin(ids_set)].copy()
len(work_ids_a_max_fs_3.drop_duplicates(subset=['id']))

90

In [ ]:
work_ids_a_max_fs_3.head(1)

,id,doi,title,type,publication_year,max_score,abstract,topic,field,subfield,...,institution_id,author_id,author,funder_id,funder,institution,problem_kw,concept_kw,context_kw,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",article,2019,1.0,Significance Electrolysis of water to generate...,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",...,97018004,5073393938,Michael Kenney,4320306076,National Science Foundation,Stanford University,[water scarcity],[nickel],[renewable energy],"[nickel, renewable energy, water scarcity]"


In [ ]:
area_do_conhecimento=work_ids_a_max_fs_3.drop_duplicates(subset=['id','field'])
area_do_conhecimento[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
4,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Aqueous Zinc-Ion Battery Technology,Engineering,Electrical and Electronic Engineering,United States,"[nickel, renewable energy, water scarcity]"
114,2153662259,10.1115/1.1737771,The Myth of Sustainable Development: Personal ...,2004,0.9998,Global Methane Emissions and Impacts,Environmental Science,Global and Planetary Change,United States,"[copper, land use change, renewable energy]"
115,2153662259,10.1115/1.1737771,The Myth of Sustainable Development: Personal ...,2004,0.9998,Global Energy Transition and Fossil Fuel Deple...,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[copper, land use change, renewable energy]"
116,2153662259,10.1115/1.1737771,The Myth of Sustainable Development: Personal ...,2004,0.9998,Impact of Oil Price Shocks on Economy,"Economics, Econometrics and Finance",Economics and Econometrics,United States,"[copper, land use change, renewable energy]"


In [ ]:
len(area_do_conhecimento)

177

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# ===== Dados =====
count_by_field = (
    area_do_conhecimento['field']
    .value_counts()
    .reset_index()
)
count_by_field.columns = ['field', 'count']
count_by_field = count_by_field.sort_values('count', ascending=False)

# Porcentagem cumulativa
total = count_by_field['count'].sum()
count_by_field['cumulative_percentage'] = (
    count_by_field['count'].cumsum() / total * 100
)

# ===== Figura =====
fig = go.Figure()

# Barras – cinza escuro
fig.add_trace(
    go.Bar(
        x=count_by_field['field'],
        y=count_by_field['count'],
        name='Número de publicações',
        marker_color='rgba(0,0,0,0.65)',
        yaxis='y1'
    )
)

# Linha – preta tracejada com marcadores brancos contornados
fig.add_trace(
    go.Scatter(
        x=count_by_field['field'],
        y=count_by_field['cumulative_percentage'],
        name='Porcentagem cumulativa',
        mode='lines+markers+text',
        line=dict(color='black', width=2, dash='dash'),
        marker=dict(size=6, color='white', line=dict(width=1, color='black')),
        text=[f"{int(p)}%" for p in count_by_field['cumulative_percentage']],
        textposition='top center',
        yaxis='y2'
    )
)

# ===== Layout ABNT/DPCT =====
fig.update_layout(
    title='Distribuição de áreas do conhecimento e porcentagem cumulativa',
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=560,
    margin=dict(l=80, r=80, t=80, b=140),
    xaxis=dict(
        title='Área do conhecimento',
        tickangle=-45,
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis2=dict(
        title='Porcentagem cumulativa (%)',
        rangemode='tozero',
        overlaying='y',
        side='right',
        range=[0, 110],
        showgrid=False,
        linecolor='black'
    ),
    showlegend=False   # nomes dos eixos já explicam
)

fig.show()



In [ ]:
count_by_field

,field,count,cumulative_percentage
0,Engineering,45,25.423729
1,Environmental Science,32,43.502825
2,Energy,27,58.757062
3,Social Sciences,27,74.011299
4,Materials Science,11,80.225989
5,"Economics, Econometrics and Finance",10,85.875706
6,Chemical Engineering,7,89.830508
7,"Business, Management and Accounting",5,92.655367
8,Earth and Planetary Sciences,2,93.785311
9,Computer Science,2,94.915254


####Publicações por área (df)

In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = area_do_conhecimento['field'].value_counts().reset_index()
count_by_field.columns = ['Área do conhecimento', 'Contagem']

# Exibir o DataFrame
count_by_field

,Área do conhecimento,Contagem
0,Engineering,45
1,Environmental Science,32
2,Energy,27
3,Social Sciences,27
4,Materials Science,11
5,"Economics, Econometrics and Finance",10
6,Chemical Engineering,7
7,"Business, Management and Accounting",5
8,Earth and Planetary Sciences,2
9,Computer Science,2


In [ ]:
count_by_field['Contagem'].sum()

np.int64(177)

###Distribuição de publicação por área ao longo do tempo

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# 1) Contar publicações por ano e área
publications_per_year_field = (
    area_do_conhecimento
    .groupby(['publication_year', 'field'])
    .size()
    .reset_index(name='count')
)

# 2) Pivot: linhas = anos, colunas = áreas; valores = contagem cumulativa
pivot_df = publications_per_year_field.pivot_table(
    index='publication_year',
    columns='field',
    values='count',
    aggfunc='sum',
    fill_value=0
).sort_index().cumsum()

# 3) Ordenar colunas por total final (maiores primeiro)
pivot_df = pivot_df[pivot_df.iloc[-1].sort_values(ascending=False).index]

# 4) Marcadores distintos
marker_symbols = [
    'circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
    'triangle-left', 'triangle-right', 'cross', 'x', 'star',
    'pentagon', 'hexagon', 'hourglass', 'bowtie', 'asterisk'
]
symbol_cycle = cycle(marker_symbols)

# 5) Figura Plotly no padrão ABNT/DPCT
fig = go.Figure()

for col in pivot_df.columns:
    fig.add_trace(
        go.Scatter(
            x=pivot_df.index.astype(str),
            y=pivot_df[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.8, dash='solid'),
            marker=dict(
                symbol=next(symbol_cycle),
                size=7,
                color='white',
                line=dict(color='black', width=1)
            ),
            hovertemplate="Ano: %{x}<br>Acumulado: %{y}<extra>"+str(col)+"</extra>"
        )
    )

# 6) Layout com eixo Y logarítmico
fig.update_layout(
    title="Volume acumulado de publicações por ano, por área do conhecimento",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    xaxis=dict(
        title="Ano de publicação",
        type='category',
        categoryorder='category ascending',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Número acumulado de publicações",
        type='linear',                # <<< escala logarítmica
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# 1) Contar publicações por ano e área
publications_per_year_field = (
    area_do_conhecimento
    .groupby(['publication_year', 'field'])
    .size()
    .reset_index(name='count')
)

# 2) Pivot: linhas = anos, colunas = áreas; valores = contagem cumulativa
pivot_df = publications_per_year_field.pivot_table(
    index='publication_year',
    columns='field',
    values='count',
    aggfunc='sum',
    fill_value=0
).sort_index().cumsum()

# 3) Ordenar colunas por total final (maiores primeiro)
pivot_df = pivot_df[pivot_df.iloc[-1].sort_values(ascending=False).index]

# 4) Marcadores distintos
marker_symbols = [
    'circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
    'triangle-left', 'triangle-right', 'cross', 'x', 'star',
    'pentagon', 'hexagon', 'hourglass', 'bowtie', 'asterisk'
]
symbol_cycle = cycle(marker_symbols)

# 5) Figura Plotly (stacked em %)
fig = go.Figure()

for col in pivot_df.columns:
    fig.add_trace(
        go.Scatter(
            x=pivot_df.index.astype(int),  # garantir que seja numérico
            y=pivot_df[col],
            mode='lines+markers',
            name=str(col),
            stackgroup='one',           # empilhamento
            groupnorm='percent',        # normalizar para 100 %
            fill='none',                # sem preenchimento cinza
            line=dict(color='black', width=1.5, dash='solid'),
            marker=dict(
                symbol=next(symbol_cycle),
                size=7,
                color='white',
                line=dict(color='black', width=1)
            ),
            hovertemplate="Ano: %{x}<br>% da área: %{y:.1f}%<extra>"+str(col)+"</extra>"
        )
    )

# === 6) Layout ABNT/DPCT garantindo início no primeiro ano ===
first_year = int(pivot_df.index.min())
last_year  = int(pivot_df.index.max())

fig.update_layout(
    title="Proporção acumulada de publicações por ano, por área do conhecimento",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    paper_bgcolor='white',
    plot_bgcolor='white',
    xaxis=dict(
        title="Ano de publicação",
        type='linear',                       # eixo numérico
        range=[first_year, last_year],       # <<< começa no primeiro ano real
        dtick=1,                              # um tick por ano
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title="Percentual (%)",
        range=[0, 100],
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


###Publicações por país

In [ ]:
paises=work_ids_a_max_fs_3.drop_duplicates(subset=['id','country'])
paises[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
18,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",China,"[nickel, renewable energy, water scarcity]"
78,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",Taiwan,"[nickel, renewable energy, water scarcity]"
114,2153662259,10.1115/1.1737771,The Myth of Sustainable Development: Personal ...,2004,0.9998,Global Methane Emissions and Impacts,Environmental Science,Global and Planetary Change,United States,"[copper, land use change, renewable energy]"
120,4379259799,10.1111/1467-8675.12686,The Problem with the Anthropocene: <i>Kainos</...,2023,0.9795,Geoengineering and Climate Ethics,Environmental Science,Global and Planetary Change,Ireland,"[cobalt, human rights, lithium, nickel, renewa..."


In [ ]:
len(paises)

132

In [ ]:
import plotly.graph_objects as go

# Contar ocorrência de cada país
count_by_country = paises['country'].value_counts().reset_index()
count_by_country.columns = ['country', 'count']
count_by_country = count_by_country.sort_values('count', ascending=False)

# Porcentagem cumulativa
total = count_by_country['count'].sum()
count_by_country['cumulative_percentage'] = (
    count_by_country['count'].cumsum() / total * 100
)

# Top 20
top20 = count_by_country.head(20)

# Figura
fig = go.Figure()

# Barras — cinza escuro
fig.add_trace(
    go.Bar(
        x=top20['country'],
        y=top20['count'],
        name='Número de publicações',
        marker_color='rgba(0,0,0,0.65)',
        yaxis='y1'
    )
)

# Linha — preta tracejada, marcadores brancos contornados
fig.add_trace(
    go.Scatter(
        x=top20['country'],
        y=top20['cumulative_percentage'],
        name='Porcentagem cumulativa',
        mode='lines+markers+text',
        line=dict(color='black', width=2, dash='dash'),
        marker=dict(size=6, color='white', line=dict(width=1, color='black')),
        text=[f"{int(p)}%" for p in top20['cumulative_percentage']],
        textposition='top center',
        yaxis='y2'
    )
)

# Layout ABNT/DPCT
fig.update_layout(
    title='Top 20 países por número de publicações e porcentagem cumulativa',
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=560,
    margin=dict(l=80, r=80, t=80, b=120),
    xaxis=dict(
        title='País',
        tickangle=-45,
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis2=dict(
        title='Porcentagem cumulativa (%)',
        rangemode='tozero',
        overlaying='y',
        side='right',
        range=[0, 110],
        showgrid=False,
        linecolor='black'
    ),
    bargap=0.2,
    showlegend=False
)

fig.show()


In [ ]:
top20

,country,count,cumulative_percentage
0,United States,19,17.592593
1,China,7,24.074074
2,Germany,6,29.629630
3,Pakistan,6,35.185185
4,United Kingdom,6,40.740741
5,Norway,6,46.296296
6,Spain,4,50.000000
7,Australia,4,53.703704
8,South Korea,4,57.407407
9,Portugal,3,60.185185


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = paises['field'].value_counts().reset_index()
count_by_field.columns = ['Área do conhecimento', 'Contagem']

# Exibir o DataFrame
count_by_field

,Área do conhecimento,Contagem
0,Engineering,49
1,Social Sciences,26
2,Environmental Science,21
3,Energy,20
4,Materials Science,7
5,Computer Science,2
6,"Economics, Econometrics and Finance",2
7,Chemical Engineering,2
8,Arts and Humanities,1
9,"Business, Management and Accounting",1


In [ ]:
count_by_field['Contagem'].sum()

np.int64(132)

###Publicações por ano

In [ ]:
ano=work_ids_a_max_fs_3.drop_duplicates(subset=['id'])
ano[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0000,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
114,2153662259,10.1115/1.1737771,The Myth of Sustainable Development: Personal ...,2004,0.9998,Global Methane Emissions and Impacts,Environmental Science,Global and Planetary Change,United States,"[copper, land use change, renewable energy]"
120,4379259799,10.1111/1467-8675.12686,The Problem with the Anthropocene: <i>Kainos</...,2023,0.9795,Geoengineering and Climate Ethics,Environmental Science,Global and Planetary Change,Ireland,"[cobalt, human rights, lithium, nickel, renewa..."
123,4316672621,10.3390/su15021736,"Spatial Planning, Environmental Activism, and ...",2023,0.9941,Sustainable Development in Mining and Minerals...,Engineering,Building and Construction,Serbia,"[environmental activism, lithium, public parti..."
132,4389135187,10.2458/jpe.5660,Title Pending 5660,2023,0.0523,Intellectual Property Law and Policy Analysis,Social Sciences,Law,Ecuador,"[climate coloniality, copper, energy transitio..."


In [ ]:
len(ano)

90

In [ ]:
import pandas as pd
import plotly.express as px

# ===================== 1) Data =====================
publications_per_year = (
    ano['publication_year']
    .value_counts()
    .reset_index()
)
publications_per_year.columns = ['publication_year', 'count']
publications_per_year = publications_per_year.sort_values('publication_year')

# make sure the year is numeric
publications_per_year['publication_year'] = publications_per_year['publication_year'].astype(int)

first_year = int(publications_per_year['publication_year'].min())
last_year  = int(publications_per_year['publication_year'].max())

# ===================== 2) Figure =====================
fig = px.line(
    publications_per_year,
    x='publication_year',
    y='count',
    markers=True,
    title='Volume de publicações por ano',
    labels={
        'publication_year': 'Ano de publicação',
        'count': 'Número de publicações'
    }
)

fig.update_traces(
    line=dict(color='black', width=2),
    marker=dict(size=6, color='white', line=dict(width=1, color='black'))
)

# ===================== 3) Layout =====================
fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    height=560,
    margin=dict(l=80, r=80, t=80, b=80),
    xaxis=dict(
        title='Ano de publicação',
        type='linear',                      # treat as numeric
        range=[first_year, last_year],      # start at first year
        dtick=1,                             # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Número de publicações',
        rangemode='tozero',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    )
)

fig.show()


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_year = ano['publication_year'].value_counts().reset_index()

# Exibir o DataFrame
count_by_year

,publication_year,count
0,2023,30
1,2022,18
2,2021,7
3,2016,6
4,2018,5
5,2020,5
6,2019,5
7,2014,4
8,2011,2
9,2015,2


In [ ]:
count_by_year['count'].sum()

np.int64(90)

###Publicações por país/ano

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from itertools import cycle

# ===== 1) Dados =====
publications_per_year_country = (
    paises
    .groupby(['publication_year', 'country'])
    .size()
    .reset_index(name='count')
)

# Pivot (anos x países) e cumulativo
pivot_df = (
    publications_per_year_country
    .pivot_table(index='publication_year', columns='country', values='count', aggfunc='sum', fill_value=0)
    .sort_index()
    .cumsum()
)

# Selecionar Top 20 países pelo total acumulado
top_countries = pivot_df.iloc[-1].sort_values(ascending=False).head(20).index
pivot_df = pivot_df[top_countries]

# ===== 2) Estilos =====
marker_symbols = ['circle', 'square', 'diamond', 'triangle-up', 'triangle-down',
                  'triangle-left', 'triangle-right', 'cross', 'x', 'star']
dash_styles    = ['solid', 'dash']
styles = cycle([(m, d) for d in dash_styles for m in marker_symbols])

# ===== 3) Figura Plotly =====
fig = go.Figure()

for col in pivot_df.columns:
    marker_sym, dash = next(styles)
    fig.add_trace(
        go.Scatter(
            x=pivot_df.index,   # keep as numeric for a true numeric axis
            y=pivot_df[col],
            mode='lines+markers',
            name=str(col),
            line=dict(color='black', width=1.5, dash=dash),
            marker=dict(symbol=marker_sym, size=6,
                        color='white', line=dict(color='black', width=1)),
            hovertemplate="Ano: %{x}<br>Acumulado: %{y}<extra>"+str(col)+"</extra>"
        )
    )

# ===== 4) Layout ABNT/DPCT-IG =====
first_year = int(pivot_df.index.min())
last_year  = int(pivot_df.index.max())

fig.update_layout(
    title="Volume acumulado de publicações por ano – Top 20 países",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    xaxis=dict(
        title="Ano de publicação",
        type='linear',                # continuous numeric axis
        range=[first_year, last_year],# start exactly at the first available year
        dtick=1,                      # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title="Número acumulado de publicações",
        rangemode='tozero',
        type='linear',
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top', y=-0.25,
        xanchor='center', x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from itertools import cycle

# ==============================================
# 1) Prepare data
# ==============================================
publications_per_year_country = (
    paises
    .groupby(['publication_year', 'country'])
    .size()
    .reset_index(name='count')
)

pivot_df = publications_per_year_country.pivot_table(
    index='publication_year',
    columns='country',
    values='count',
    aggfunc='sum',
    fill_value=0
)

# Keep top 20 contributors
top_countries = pivot_df.sum().nlargest(20).index
pivot_df = pivot_df[top_countries].sort_index().cumsum()

# Percent of total per year (stacked 100 %)
pivot_df_percent = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

# Order by final contribution
country_order = pivot_df.iloc[-1].sort_values(ascending=False).index.tolist()
pivot_df_percent = pivot_df_percent[country_order]

# Make sure year is numeric and sorted
pivot_df_percent.index = pivot_df_percent.index.astype(int)
first_year = int(pivot_df_percent.index.min())
last_year  = int(pivot_df_percent.index.max())

# ==============================================
# 2) Define line+marker styles
# ==============================================
marker_symbols = ['circle','square','diamond','triangle-up','triangle-down',
                  'triangle-left','triangle-right','cross','x','star']
dash_styles    = ['solid','dash']
styles = cycle([(m, d) for d in dash_styles for m in marker_symbols])

# ==============================================
# 3) Build stacked 100 % line/area chart
# ==============================================
fig = go.Figure()

for col in pivot_df_percent.columns:
    marker_sym, dash = next(styles)
    fig.add_trace(
        go.Scatter(
            x=pivot_df_percent.index,
            y=pivot_df_percent[col],
            mode='lines+markers',
            name=col,
            stackgroup='one',           # stack to 100 %
            groupnorm='percent',
            fill='none',                # transparent fill
            line=dict(color='black', width=1.5, dash=dash),
            marker=dict(symbol=marker_sym, size=6,
                        color='white', line=dict(color='black', width=1)),
            hoverinfo='skip'            # <<< no hover boxes
        )
    )

# ==============================================
# 4) Layout in ABNT/DPCT style
# ==============================================
fig.update_layout(
    title="Proporção acumulada de publicações por ano – Top 20 países",
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    height=650,
    margin=dict(l=80, r=80, t=80, b=160),
    paper_bgcolor='white',   # outer background
    plot_bgcolor='white',    # plotting area background
    xaxis=dict(
        title='Ano de publicação',
        type='linear',                   # numeric axis
        range=[first_year, last_year],   # start at first actual year
        dtick=1,                         # tick every year
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    yaxis=dict(
        title='Percentual acumulado (%)',
        range=[0, 100],
        showgrid=True,
        gridcolor='rgba(0,0,0,0.1)',
        linecolor='black',
        mirror=True
    ),
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.25,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)'
    )
)

fig.show()


In [ ]:
import pandas as pd

# Supondo que o dataframe já esteja carregado
# work_ids_a_max_fs_yr_3 = pd.read_csv('seu_arquivo.csv')

# Contar a ocorrência de cada área do conhecimento
count_by_field = work_ids_a_max_fs_3['publication_year'].value_counts().reset_index()
count_by_field.columns = ['Ano de publicação', 'Contagem']

# Exibir o DataFrame
print(count_by_field)

    Ano de publicação  Contagem
0                2023       776
1                2020       235
2                2022       229
3                2019       124
4                2017        36
5                2016        33
6                2018        31
7                2014        25
8                2021        23
9                2015        21
10               2013        12
11               2010        12
12               2004         6
13               2011         6
14               2008         1


In [ ]:
count_by_field['Contagem'].sum()

np.int64(1570)

###Publicações por instituição

In [ ]:
institution=work_ids_a_max_fs_3.drop_duplicates(subset=['id','institution'])
institution[['id','doi','title','publication_year','max_score','topic','field','subfield','country','author','institution','funder','all_keywords']].head(5)

,id,doi,title,publication_year,max_score,topic,field,subfield,country,author,institution,funder,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,Michael Kenney,Stanford University,National Science Foundation,"[nickel, renewable energy, water scarcity]"
18,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",China,Yongtao Meng,Shandong University of Science and Technology,National Science Foundation,"[nickel, renewable energy, water scarcity]"
24,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",China,Xiaoming Sun,Beijing University of Chemical Technology,National Science Foundation,"[nickel, renewable energy, water scarcity]"
36,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,Yijin Liu,SLAC National Accelerator Laboratory,National Science Foundation,"[nickel, renewable energy, water scarcity]"
54,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",China,Lei Wang,Tianjin University of Technology,National Science Foundation,"[nickel, renewable energy, water scarcity]"


In [ ]:
len(institution)

187

In [ ]:
institution_count=institution['institution'].value_counts().reset_index()
institution_count.columns = ['institution', 'count']
institution_count = institution_count.sort_values(by='count', ascending=False)
institution_count

,institution,count
0,National University of Sciences and Technology,4
1,Stanford University,2
2,Korea Institute of Industrial Technology,2
3,Tianjin University of Technology,2
4,Konkuk University,2
...,...,...
141,Erasmus University Rotterdam,1
142,Patuakhali Science and Technology University,1
143,Lawrence Berkeley National Laboratory,1
144,"University of California, Davis",1


In [ ]:
import plotly.express as px
import numpy as np

TOP_N = 20
plot_df = institution_count.head(TOP_N)

# paleta em tons de cinza (do mais escuro para o mais claro)
grays = [f"rgb({v},{v},{v})" for v in np.linspace(30, 200, num=TOP_N, dtype=int)]

fig = px.bar(
    plot_df,
    x="count",
    y="institution",
    orientation="h",
    text="count",
    labels={"count": "Número de publicações", "institution": "Instituição"},
    title=f"Top {TOP_N} instituições por número de publicações",
    color_discrete_sequence=grays
)

fig.update_traces(
    textposition="outside",
    cliponaxis=False,
    marker_line_width=1,
    marker_line_color="black"
)

fig.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=12),
    showlegend=False,
    xaxis=dict(
        title='Número de publicações',
        showgrid=True, gridcolor='rgba(0,0,0,0.1)',
        linecolor='black', mirror=True
    ),
    yaxis=dict(
        title='Instituição',
        categoryorder='total ascending',  # maior valor no topo
        showgrid=False,
        linecolor='black', mirror=True
    ),
    margin=dict(l=220, r=40, t=80, b=60),  # espaço extra para nomes longos
    height=700
)

fig.show()


##WIP

In [ ]:
df=work_ids_a_max_fs_3
df[['id','doi','title','publication_year','max_score','topic','field','subfield','country','all_keywords']].head()

,id,doi,title,publication_year,max_score,topic,field,subfield,country,all_keywords
0,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
1,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Electrocatalysis for Energy Conversion,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
2,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Photocatalytic Materials for Solar Energy Conv...,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
3,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Photocatalytic Materials for Solar Energy Conv...,Energy,"Renewable Energy, Sustainability and the Envir...",United States,"[nickel, renewable energy, water scarcity]"
4,2922256802,10.1073/pnas.1900556116,"Solar-driven, highly sustained splitting of se...",2019,1.0,Aqueous Zinc-Ion Battery Technology,Engineering,Electrical and Electronic Engineering,United States,"[nickel, renewable energy, water scarcity]"


In [ ]:
%%bigquery work_ids_a_max_fs_wip --project insyspo

SELECT a.*, f.abstract as abstract, c.display_name as topic, e.display_name AS field, d.display_name as subfield, i.country, h.work_id,
institution_id, h.author_id, j.display_name as author, k.funder_id, l.display_name as funder, i.display_name as institution
FROM insyspo.userdb_fernando_spolidorio.cm_work_ids_a_maxscores_3 AS a
JOIN insyspo.publicdb_openalex_2024_10_rm.works_topics AS b ON a.id = b.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.topics AS c ON b.topic_id = c.id
JOIN insyspo.publicdb_openalex_2024_10_rm.subfields AS d ON c.subfield = d.id
JOIN insyspo.publicdb_openalex_2024_10_rm.fields AS e ON c.field = e.id
JOIN insyspo.projectdb_cwts_summer_school_2023.works_abstracts f ON a.id = f.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works g ON a.id = g.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works_authorships AS h ON a.id = h.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.institutions AS i ON h.institution_id = i.id
JOIN insyspo.publicdb_openalex_2024_10_rm.authors AS j ON h.author_id = j.id
JOIN insyspo.publicdb_openalex_2024_10_rm.works_grants AS k ON a.id = k.work_id
JOIN insyspo.publicdb_openalex_2024_10_rm.funders AS l ON k.funder_id = l.id




ERROR:
 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/insyspo/jobs?prettyPrint=false: Access Denied: Project insyspo: User does not have bigquery.jobs.create permission in project insyspo.

Location: None
Job ID: f93190ca-271e-42ca-b598-a12885ce153b



In [ ]:
df=work_ids_a_max_fs_wip
df[['id','doi','title','publication_year','max_score','topic','field','subfield','country','author','funder','institution']].head()

TypeError: 'QueryJob' object is not subscriptable

In [ ]:
author_works = (
    df.groupby('author')
      .apply(lambda x: list(zip(x['id'], x['title'])))
      .reset_index(name='works')
)

# add count as well
author_works['count'] = author_works['works'].apply(len)

# sort by count
author_works = author_works.sort_values('count', ascending=False)

author_works


#WIP

In [ ]:
import pandas as pd

# 1) Read as strings
coords = pd.read_csv(
    '/content/drive/MyDrive/cientometria_resultados/Country networks (Gephi)/countries latlong 2.csv',
    dtype=str
)

# 2) Function to fix each coordinate value
def fix_coord(value):
    if pd.isna(value):
        return None
    # Remove all dots
    digits = value.replace('.', '')
    # If too short, return NaN
    if len(digits) <= 6:
        return None
    # Insert decimal before last 6 digits
    fixed = digits[:-6] + '.' + digits[-6:]
    return float(fixed)

# 3) Apply to both columns
coords['latitude'] = coords['latitude'].apply(fix_coord)
coords['longitude'] = coords['longitude'].apply(fix_coord)

# 4) Inspect
print(coords.head())


  country   latitude  longitude                  name
0      AD  42.546245   1.601554               Andorra
1      AE  23.424076  53.847818  United Arab Emirates
2      AF   3.393911  67.709953           Afghanistan
3      AG  17.060816 -61.796428   Antigua and Barbuda
4      AI  18.220554 -63.068615              Anguilla


In [ ]:
# ✅ Step 1: Clean coords (done previously)
# coords now has clean 'latitude', 'longitude', 'name'

# ✅ Step 2: Merge on country = name
merged_df = filtered_df_sorted.merge(
    coords[['name', 'latitude', 'longitude']],
    left_on='country',
    right_on='name',
    how='left'   # use 'left' to keep all rows from filtered_df_sorted
)

# ✅ Step 3: Drop redundant 'name' column if you don’t need it
merged_df = merged_df.drop(columns=['name'])

merged_df


,id,country,work_id,author_id,raw_affiliation_string,author_position,author_order,institution_id,latitude,longitude
0,25084758,United States,25084758,5102412187,None,last,3,2799863703,3.709024,-95.712891
1,25084758,United States,25084758,5050489657,None,first,1,4210103952,3.709024,-95.712891
2,34317772,Australia,34317772,5047994012,None,last,3,118347636,-25.274398,133.775136
3,34317772,Australia,34317772,5030590660,None,first,1,82951845,-25.274398,133.775136
4,34317772,Australia,34317772,5009264809,None,middle,2,149704539,-25.274398,133.775136
...,...,...,...,...,...,...,...,...,...,...
29426,4402568861,Taiwan,4402568861,5021656897,None,middle,3,107470533,2.369781,120.960515
29427,4402568861,India,4402568861,5003580682,None,first,1,85461943,20.593684,7.896288
29428,4402568861,China,4402568861,5029139156,None,middle,5,87445476,3.586166,104.195397
29429,4402568861,India,4402568861,5011900641,None,middle,2,85461943,20.593684,7.896288


##Gephi

In [ ]:
import pandas as pd
from itertools import combinations

# === 1) Load your data ===
# merged_df must have: 'id' and 'country'

# Example:
# merged_df = pd.read_csv('your_merged_df.csv')

merged_df=work_ids_a_max_fs_3

# === 2) Get unique countries per ID ===
id_countries = merged_df.groupby('id')['country'].unique().reset_index()

# === 3) Build all unordered pairs within each ID ===
pairs = []
for row in id_countries.itertuples(index=False):
    countries = row.country
    if len(countries) < 2:
        continue
    for pair in combinations(sorted(countries), 2):
        pairs.append(pair)

# === 4) Make pairs DataFrame ===
pairs_df = pd.DataFrame(pairs, columns=['Source', 'Target'])

# === 5) Count co-occurrences ===
edges = pairs_df.groupby(['Source', 'Target']).size().reset_index(name='Weight')

# === 6) OPTIONAL: Filter weak edges ===
# THRESHOLD = 5  # 👈 tweak this!
# edges = edges[edges['Weight'] >= THRESHOLD]

# Or limit to top N strongest pairs:
# TOP_N = 200
# edges = edges.sort_values('weight', ascending=False).head(TOP_N)

# === 7) Save final edges for Kepler or Gephi ===
# edges.to_csv(
#     '/content/drive/MyDrive/cientometria_resultados/Country networks (Gephi)/edges_countries.csv',
#     index=False
# )

edges


,Source,Target,Weight
0,Albania,China,1
1,Algeria,Austria,1
2,Algeria,Brunei,1
3,Algeria,Ecuador,1
4,Algeria,Egypt,1
...,...,...,...
787,United Kingdom,Venezuela,1
788,United States,Venezuela,1
789,United States,Vietnam,2
790,United States,Zambia,1


In [ ]:
test=work_ids_a_max_fs_3.drop_duplicates(subset=['id', 'institution'])

import pandas as pd
from itertools import combinations

# 1. garantir pares únicos id–institution
df = work_ids_a_max_fs_3[['id', 'institution']].drop_duplicates()

# 2. gerar pares de instituições por id
edges = (
    df.groupby('id')['institution']
      .apply(lambda x: list(combinations(sorted(x.unique()), 2)))
      .explode()
      .dropna()
)

# 3. transformar em DataFrame
edges_df = pd.DataFrame(edges.tolist(), columns=['source', 'target'])

# 4. calcular peso (frequência da colaboração)
edges_df = (
    edges_df
    .value_counts()
    .reset_index(name='weight')
)

edges_df[edges_df['weight']>=3]



,source,target,weight
0,Chinese Academy of Sciences,University of Chinese Academy of Sciences,33
1,Oak Ridge National Laboratory,University of Tennessee at Knoxville,17
2,Chinese Academy of Sciences,Dalian Institute of Chemical Physics,15
3,Argonne National Laboratory,Los Alamos National Laboratory,13
4,Lawrence Berkeley National Laboratory,"University of California, Berkeley",11
...,...,...,...
211,Chinese Academy of Sciences,Fuzhou University,3
212,Academy of Scientific and Innovative Research,Central Electrochemical Research Institute,3
213,Carnegie Mellon University,Giner (United States),3
214,Argonne National Laboratory,"University at Buffalo, State University of New...",3


In [ ]:
import pandas as pd
from itertools import combinations

# === 1) Load your data ===
# merged_df must have: 'id' and 'country'
# coords must have: 'name', 'latitude', 'longitude'

# Example:
# merged_df = pd.read_csv('your_merged_df.csv')
# coords = pd.read_csv('your_coords.csv')

# === 2) Get unique countries per ID ===
id_countries = merged_df.groupby('id')['country'].unique().reset_index()

# === 3) Build all unordered pairs within each ID ===
pairs = []
for row in id_countries.itertuples(index=False):
    countries = row.country
    if len(countries) < 2:
        continue
    for pair in combinations(sorted(countries), 2):
        pairs.append(pair)

# === 4) Make pairs DataFrame ===
pairs_df = pd.DataFrame(pairs, columns=['source', 'target'])

# === 5) Count co-occurrences ===
edges = pairs_df.groupby(['source', 'target']).size().reset_index(name='weight')

# === 6) Merge source coordinates ===
edges = edges.merge(
    coords[['name', 'latitude', 'longitude']].rename(
        columns={'name': 'source', 'latitude': 'lat_source', 'longitude': 'lon_source'}
    ),
    on='source',
    how='left'
)

# === 7) Merge target coordinates ===
edges = edges.merge(
    coords[['name', 'latitude', 'longitude']].rename(
        columns={'name': 'target', 'latitude': 'lat_target', 'longitude': 'lon_target'}
    ),
    on='target',
    how='left'
)

# === 8) Keep only needed columns ===
edges_geo = edges[['lat_source', 'lon_source', 'lat_target', 'lon_target', 'weight']]

# === 9) OPTIONAL: Filter weak edges ===
# Keep only edges with weight >= threshold
THRESHOLD = 5  # 👈 tweak this!
edges_geo = edges_geo[edges_geo['weight'] >= THRESHOLD]

# Or limit to top N strongest connections:
# TOP_N = 200
# edges_geo = edges_geo.sort_values('weight', ascending=False).head(TOP_N)

# === 10) Save final edges for Kepler ===
edges_geo.to_csv(
    '/content/drive/MyDrive/cientometria_resultados/Country networks (Gephi)/edges_clean.csv',
    index=False
)

# === 11) Check ===
print(edges_geo.head())
print(f"Total edges exported: {len(edges_geo)}")


    lat_source  lon_source  lat_target  lon_target  weight
15   28.033886    1.659626   46.227638    2.213749       6
59  -25.274398  133.775136   23.684994   90.356331       6
62  -25.274398  133.775136  -14.235004   -5.192528       5
65  -25.274398  133.775136   56.130366 -106.346771       8
66  -25.274398  133.775136  -35.675147  -71.542969       7
Total edges exported: 187


In [ ]:
edges.to_csv('/content/drive/MyDrive/cientometria_resultados/Country networks (Gephi)/edges.csv', index=False)

In [ ]:
# Get unique nodes from both source and target
nodes_sources = edges[['source']].rename(columns={'source': 'id'})
nodes_targets = edges[['target']].rename(columns={'target': 'id'})
nodes_all = pd.concat([nodes_sources, nodes_targets]).drop_duplicates()

# Join back lat/long from merged_df (only once per id)
nodes = nodes_all.merge(
    merged_df[['country', 'latitude', 'longitude']],
    left_on='id',
    right_on='country',
    how='left'
).drop(columns=['country']).drop_duplicates()

nodes.to_csv('/content/drive/MyDrive/cientometria_resultados/Country networks (Gephi)/nodes.csv', index=False)